In [2]:
# Cell 1 — Setup: packages, settings, palette, helpers, Google Drive
# =====================================================================
# SIM LAB  ·  a small computational-imaging project (runs in ~15 min on a free Colab T4)
# Output: ONE self-contained HTML dashboard  ->  Google Drive (MyDrive/SIM_Lab/)
# =====================================================================
import os, sys, io, time, json, math, base64, warnings, subprocess, importlib, platform
warnings.filterwarnings("ignore")

def _ensure(pkg, imp=None):
    try:
        importlib.import_module(imp or pkg)
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg], check=False)

for _p, _i in [("scikit-image", "skimage"), ("pooch", "pooch"), ("plotly", "plotly"),
               ("pillow", "PIL"), ("scipy", "scipy"), ("pandas", "pandas"), ("PyWavelets", "pywt")]:
    _ensure(_p, _i)

import numpy as np, pandas as pd
import scipy.fft as sfft
from scipy import ndimage as ndi
import torch, torch.nn as nn, torch.nn.functional as F
import plotly, plotly.graph_objects as go
from plotly.subplots import make_subplots
from PIL import Image
from skimage import data as skdata
from skimage.metrics import structural_similarity as ssim_fn
from skimage.restoration import estimate_sigma

# ------------------------------------------------------------ user settings
AUTHOR      = "Alaa Guedda"          # <- shown in the dashboard header (edit me)
QUICK_MODE  = False                # True = 3-minute smoke test (smaller sweeps / training)
SEED        = 7
OUT_NAME    = "sim_lab_dashboard.html"
DRIVE_DIR   = "SIM_Lab"            # folder created inside My Drive
np.random.seed(SEED); torch.manual_seed(SEED)

# ------------------------------------------------------------ Drive + device
IN_COLAB = "google.colab" in sys.modules
OUT_DIR = "/content" if IN_COLAB else os.getcwd()
if IN_COLAB:
    try:
        from google.colab import drive
        drive.mount("/content/drive")
        OUT_DIR = f"/content/drive/MyDrive/{DRIVE_DIR}"
        os.makedirs(OUT_DIR, exist_ok=True)
    except Exception as e:
        print("Drive not mounted, saving to /content instead:", e)
        OUT_DIR = "/content"
DEV = "cuda" if torch.cuda.is_available() else "cpu"
print(f"device={DEV} | torch {torch.__version__} | plotly {plotly.__version__} | output -> {OUT_DIR}")
if DEV == "cpu":
    print("WARNING: no GPU found. Runtime > Change runtime type > T4 GPU (training will be slow on CPU).")

T_START = time.time()
def log(msg): print(f"[{time.time()-T_START:6.0f}s] {msg}")

# ------------------------------------------------------------ palette (light mode)
C = dict(g1="#1dad22", g2="#268c32", g3="#37bf53", y="#e0d01f", d="#494b52", l="#bdbebf")
def mix(a, b="#ffffff", t=0.5):
    ca = [int(a[i:i+2], 16) for i in (1, 3, 5)]; cb = [int(b[i:i+2], 16) for i in (1, 3, 5)]
    return "#" + "".join(f"{int(round(x*(1-t)+y*t)):02x}" for x, y in zip(ca, cb))
GRID = mix(C["l"], "#ffffff", 0.6)
FONT = "Inter, system-ui, -apple-system, 'Segoe UI', Roboto, Helvetica, Arial, sans-serif"
SCALE_Q   = [[0, C["l"]], [0.45, C["y"]], [0.75, C["g3"]], [1, C["g2"]]]        # quality: grey -> yellow -> green
SCALE_SEQ = [[0, "#f6faf6"], [0.5, C["g3"]], [1, C["g2"]]]
SCALE_DIV = [[0, C["d"]], [0.5, "#ffffff"], [1, C["g1"]]]
SCALE_SPEC = [[0, "#ffffff"], [0.35, mix(C["g3"], "#ffffff", 0.55)], [0.7, C["g1"]], [1, C["d"]]]

def style(fig, h=420, legend=True, title=None):
    fig.update_layout(
        template="plotly_white", font=dict(family=FONT, size=13, color=C["d"]), height=h,
        margin=dict(l=58, r=24, t=56 if title else 30, b=54),
        paper_bgcolor="rgba(0,0,0,0)", plot_bgcolor="rgba(0,0,0,0)",
        colorway=[C["g1"], C["y"], C["d"], C["g3"], C["g2"], C["l"]],
        hoverlabel=dict(bgcolor="white", bordercolor=C["l"], font=dict(color=C["d"], family=FONT)),
        legend=dict(orientation="h", y=1.14, x=0, font=dict(size=12)) if legend else None,
        showlegend=legend,
        title=dict(text=title, x=0.01, font=dict(size=15)) if title else None)
    fig.update_xaxes(gridcolor=GRID, zerolinecolor=GRID, linecolor=C["l"], tickcolor=C["l"], ticks="outside")
    fig.update_yaxes(gridcolor=GRID, zerolinecolor=GRID, linecolor=C["l"], tickcolor=C["l"], ticks="outside")
    return fig

def fig_html(fig, div_id):
    return fig.to_html(full_html=False, include_plotlyjs=False, div_id=div_id,
                       config=dict(responsive=True, displaylogo=False, modeBarButtonsToRemove=["lasso2d", "select2d"]))

# ------------------------------------------------------------ small helpers
def norm01(a, lo=0.5, hi=99.8):
    a = np.asarray(a, dtype=np.float32); l, h = np.percentile(a, [lo, hi])
    return np.clip((a - l) / (h - l + 1e-9), 0, 1)

def img_uri(arr01, fmt="WEBP", q=93):
    im = Image.fromarray((np.clip(arr01, 0, 1) * 255 + 0.5).astype(np.uint8), "L").convert("RGB")
    buf = io.BytesIO()
    if fmt == "PNG": im.save(buf, "PNG", optimize=True)
    else: im.save(buf, fmt, quality=q)
    return f"data:image/{fmt.lower()};base64," + base64.b64encode(buf.getvalue()).decode()

def radial_spectrum(img, nb=64):
    """Radially-averaged power spectrum of a square image. returns (freq cycles/px, power)."""
    s = min(img.shape); img = img[:s, :s].astype(np.float64)
    w = np.outer(np.hanning(s), np.hanning(s))
    P = np.abs(np.fft.fftshift(np.fft.fft2((img - img.mean()) * w))) ** 2
    f = np.fft.fftshift(np.fft.fftfreq(s)); kr = np.hypot(f[None, :], f[:, None])
    bins = np.linspace(0, 0.5, nb + 1); idx = np.digitize(kr.ravel(), bins) - 1; ok = idx < nb
    cnt = np.bincount(idx[ok], minlength=nb); tot = np.bincount(idx[ok], weights=P.ravel()[ok], minlength=nb)
    return 0.5 * (bins[1:] + bins[:-1]), tot / np.maximum(cnt, 1)

def log_spectrum2d(img):
    s = min(img.shape); img = img[:s, :s].astype(np.float64)
    w = np.outer(np.hanning(s), np.hanning(s))
    P = np.abs(np.fft.fftshift(np.fft.fft2((img - img.mean()) * w))) ** 2
    return np.log10(P + 1e-12)
log("cell 1 ready")


Mounted at /content/drive
device=cuda | torch 2.11.0+cu130 | plotly 5.24.1 | output -> /content/drive/MyDrive/SIM_Lab
[     0s] cell 1 ready


In [3]:
# Cell 2 — Public data: load, quality-check, select ground-truth images
# Rules come straight from the exploration run (spectral cut-off, saturation, size, noise).
LOADERS = {"cells3d": skdata.cells3d, "human_mitosis": skdata.human_mitosis, "kidney": skdata.kidney,
           "protein_transport": skdata.protein_transport, "cell": skdata.cell, "lily": skdata.lily}
RAW = {}
for _n, _fn in LOADERS.items():
    try:
        RAW[_n] = np.asarray(_fn()); log(f"loaded {_n:18s} {RAW[_n].shape} {RAW[_n].dtype}")
    except Exception as e:
        print(f"[FAIL] {_n}: {type(e).__name__}: {str(e)[:80]}")
for _need in ("cells3d", "human_mitosis", "kidney", "lily"):
    assert _need in RAW, f"core dataset '{_need}' could not be downloaded - check internet access in Colab"

def _frames(name, a):
    if name == "cells3d":
        z = a.shape[0] // 2; return [("membrane", a[z, 0]), ("nuclei", a[z, 1])]
    if a.ndim >= 3 and a.shape[-1] <= 4:
        out = []
        for c in range(min(a.shape[-1], 3)):
            m = a[..., c]
            while m.ndim > 2: m = m[m.shape[0] // 2]
            out.append((f"ch{c}", m))
        return out
    b = a
    while b.ndim > 2: b = b[b.shape[0] // 2]
    return [("image", b)]

def _stats(ds, ch, im):
    f = im.astype(np.float64); fn = (f - f.min()) / (np.ptp(f) + 1e-12)
    s = min(f.shape[:2], default=0); s = min(s, 256); r0, c0 = (f.shape[0] - s) // 2, (f.shape[1] - s) // 2
    fr, pw = radial_spectrum(f[r0:r0 + s, c0:c0 + s], 25)
    pw = pw / pw.max(); below = np.where(pw < 1e-4)[0]
    sig = estimate_sigma(fn)
    return dict(dataset=ds, channel=ch, shape=f"{im.shape[0]}x{im.shape[1]}", min_side=min(im.shape[:2]),
                mean=fn.mean(), sat=(f == f.max()).mean(), noise=sig, snr=fn.mean() / (sig + 1e-12),
                cutoff=(fr[below[0]] if len(below) else 0.5), hf=float(np.sum(pw[fr > 0.25]) / np.sum(pw)))

rows = []
for _n in RAW:
    for _ch, _im in _frames(_n, RAW[_n]):
        rows.append(_stats(_n, _ch, _im))
INV = pd.DataFrame(rows)
def _verdict(r):
    if r.min_side < 256: return "excluded", "image too small for a 256 px field of view"
    if r.cutoff < 0.35:  return "excluded", "no fine structure (spectrum ends below 0.35 cycles/px)"
    if r.sat >= 0.01:    return "excluded", ">1% saturated pixels"
    return "kept", "passes all checks"
INV[["status", "reason"]] = INV.apply(lambda r: pd.Series(_verdict(r)), axis=1)
INV["role"] = np.where(INV.status == "kept", np.where(INV.dataset.isin(["cells3d", "human_mitosis"]), "held-out test", "training"), "-")
INV["label"] = INV.dataset + ":" + INV.channel
print(INV[["label", "shape", "snr", "cutoff", "sat", "status", "role", "reason"]].to_string(index=False, float_format=lambda x: f"{x:.3g}"))

# ------------------------------------------------------------ ground-truth pools
def _crops(im, size=256, stride=None):
    stride = stride or size; out = []
    for y in range(0, im.shape[0] - size + 1, stride):
        for x in range(0, im.shape[1] - size + 1, stride):
            c = im[y:y + size, x:x + size]
            if c.std() > 0.05: out.append(c)
    return out

_kept = set(INV[INV.status == "kept"].label)
TRAIN, VAL = [], []
_k = RAW["kidney"]                                   # (Z, Y, X, C)
for _z in range(_k.shape[0]):
    for _c in (0, 1):
        if f"kidney:ch{_c}" not in _kept: continue
        (VAL if _z >= _k.shape[0] - 2 else TRAIN).extend(_crops(norm01(_k[_z, :, :, _c])))
_l = RAW["lily"]
for _c in (0, 1, 2):
    if f"lily:ch{_c}" not in _kept: continue
    _cr = _crops(norm01(_l[:, :, _c]), 256, 333)
    TRAIN.extend(_cr[:-1]); VAL.extend(_cr[-1:])
TRAIN = [t.astype(np.float32) for t in TRAIN]; VAL = [v.astype(np.float32) for v in VAL]

_c3 = RAW["cells3d"]; _mit = norm01(RAW["human_mitosis"])
_quads = _crops(_mit, 256, 256)
_best = max(_quads, key=lambda q: q.std()) if _quads else _mit[:256, :256]
TEST = {"Membranes": norm01(_c3[30, 0]).astype(np.float32),           # cells3d, z=30
        "Nuclei":    norm01(_c3[30, 1]).astype(np.float32),
        "HeLa mitosis": _best.astype(np.float32)}
TEST_SRC = {"Membranes": "cells3d · membrane · z=30", "Nuclei": "cells3d · nuclei · z=30", "HeLa mitosis": "human_mitosis · best 256² crop"}
for _k2, _v in TEST.items(): assert _v.shape == (256, 256), (_k2, _v.shape)
print(f"\nGround truth pools -> train {len(TRAIN)} | validation {len(VAL)} | held-out test {len(TEST)}")
log("cell 2 done")


[     1s] loaded cells3d            (60, 2, 256, 256) uint16


[     2s] loaded human_mitosis      (512, 512) uint8


[     2s] loaded kidney             (16, 512, 512, 3) uint16


[     3s] loaded protein_transport  (15, 2, 180, 183) uint8
[     3s] loaded cell               (660, 550) uint8
[     3s] loaded lily               (922, 922, 4) uint16
                  label   shape  snr  cutoff      sat   status          role                                                 reason
       cells3d:membrane 256x256 17.2    0.43 1.53e-05     kept held-out test                                      passes all checks
         cells3d:nuclei 256x256 8.47    0.47 1.53e-05     kept held-out test                                      passes all checks
    human_mitosis:image 512x512 14.4    0.43 7.63e-06     kept held-out test                                      passes all checks
             kidney:ch0 512x512 12.8     0.5  0.00184     kept      training                                      passes all checks
             kidney:ch1 512x512 12.6     0.5  0.00878     kept      training                                      passes all checks
             kidney:ch2 512x512 10.9  

In [4]:
# Cell 3 — SIM engine: optics, acquisition simulator, Wiener reconstruction, self-calibration, metrics
NPIX = 256
FC   = 0.20                      # detection cut-off (cycles/pixel): the diffraction limit
P0   = 0.85 * FC                 # nominal illumination-pattern frequency
B0   = 0.90                      # pattern modulation depth
THETA = np.deg2rad([0.0, 60.0, 120.0])
PH_NOM = 2 * np.pi * np.arange(3) / 3
_f = np.fft.fftfreq(NPIX)
KY, KX = np.meshgrid(_f, _f, indexing="ij"); KR = np.hypot(KX, KY)
YY, XX = np.meshgrid(np.arange(NPIX), np.arange(NPIX), indexing="ij")
fft2  = lambda a: sfft.fft2(a, axes=(-2, -1), workers=2)
ifft2 = lambda a: sfft.ifft2(a, axes=(-2, -1), workers=2)

def otf_of(kx, ky):              # incoherent OTF of a circular pupil
    rho = np.clip(np.hypot(kx, ky) / FC, 0, 1)
    return (2 / np.pi) * (np.arccos(rho) - rho * np.sqrt(1 - rho ** 2))
H0 = otf_of(KX, KY)
P_NOM = np.stack([P0 * np.cos(THETA), P0 * np.sin(THETA)], 1)     # (3,2) cycles/pixel
PH_ASSUMED = np.tile(PH_NOM, (3, 1))
KMAX = FC + P0
APOD = np.clip(1 - KR / KMAX, 0, 1) ** 0.6
APOD_WF = np.clip(1 - KR / FC, 0, 1) ** 0.6
GAUSS = lambda s: np.exp(-2 * np.pi ** 2 * s ** 2 * KR ** 2)
print(f"widefield cut-off {FC} cyc/px | pattern {P0:.3f} | SIM support {KMAX:.3f} -> {KMAX/FC:.2f}x | pattern period {1/P0:.1f} px")

def smooth_field(rng, sigma_px=55.0):
    f = np.real(ifft2(fft2(rng.standard_normal((NPIX, NPIX))) * GAUSS(sigma_px)))
    return f / np.abs(f).max()

def make_truth(phase_err_deg=0.0, freq_err=0.0, eta=0.0, seed=0):
    """True (hidden) illumination: pattern phases/frequencies off from nominal + smooth phase/contrast distortions.
    phase_err_deg: rms error of every pattern phase (deg) | freq_err: pattern-frequency error (cycles per field of view)
    eta: strength (0-1) of spatially varying pattern phase (up to 1.2 rad) and contrast (down to -50%)."""
    rng = np.random.RandomState(1000 + seed)
    Eph = rng.standard_normal((3, 3)); ang = rng.uniform(0, 2 * np.pi, 3)
    Ef = np.stack([np.cos(ang), np.sin(ang)], 1)
    fields = [(smooth_field(rng), smooth_field(rng)) for _ in range(3)]
    return dict(p=P_NOM + freq_err / NPIX * Ef, phi=PH_NOM[None, :] + np.deg2rad(phase_err_deg) * Eph,
                eta=eta, fields=fields)

def simulate(gt, truth, peak=100.0, ofb=0.0, noise_seed=0):
    """9 raw frames (3 orientations x 3 phases): blur(object x pattern) + haze + Poisson + read noise."""
    rng = np.random.RandomState(noise_seed)
    Fg = fft2(gt)
    bg = ofb * np.real(ifft2(Fg * GAUSS(10.0))) if ofb > 0 else 0.0        # unmodulated out-of-focus haze
    clean = np.empty((3, 3, NPIX, NPIX))
    for t in range(3):
        px, py = truth["p"][t]; g, h = truth["fields"][t]
        psi = truth["eta"] * 1.2 * g
        bmap = B0 * (1 - 0.5 * truth["eta"] * (0.5 + 0.5 * h))
        a0 = 2 * np.pi * (px * XX + py * YY)
        for j in range(3):
            ill = 1 + bmap * np.cos(a0 + truth["phi"][t, j] + psi)
            clean[t, j] = np.real(ifft2(fft2(gt * ill) * H0)) + bg
    clean = np.maximum(clean, 0)
    sc = peak / np.percentile(clean, 99.5)
    y = (rng.poisson(clean * sc) + rng.normal(0, 1.0, clean.shape)) / sc
    return y.astype(np.float32)

def _unmix(Ft, phi_row):
    Mx = np.exp(1j * np.outer(phi_row, [0, 1, -1]))
    return np.einsum("mj,jyx->myx", np.linalg.inv(Mx), Ft)           # components w_m * H(k) * S(k - m p)

def reconstruct(raw, p_as, phi_as, lam, b_as=B0, local=False, sigma_loc=14.0, aux=False):
    """Generalised Wiener SIM reconstruction. local=True adds a smooth, data-driven estimate of the
    pattern's phase/contrast across the field of view and divides it out (a simplified, flexible model)."""
    Fr = fft2(raw)
    num = np.zeros((NPIX, NPIX), complex); den = np.zeros((NPIX, NPIX)); rho_out = None
    w1 = b_as / 2
    sm = lambda z: ifft2(fft2(z) * GAUSS(sigma_loc))
    for t in range(3):
        px, py = p_as[t]
        Cw = _unmix(Fr[t], phi_as[t])
        ph = np.exp(-2j * np.pi * (px * XX + py * YY))
        c1 = ifft2(Cw[1]) * ph; cm = ifft2(Cw[2]) * np.conj(ph)
        H1 = otf_of(KX + px, KY + py); Hm = otf_of(KX - px, KY - py)
        if local:
            Mk = (H0 > 0.25) & (H1 > 0.25)
            A = ifft2(np.where(Mk, Cw[0] / np.maximum(H0, 1e-3), 0))
            B = ifft2(np.where(Mk, fft2(c1) / np.maximum(w1 * H1, 1e-3), 0))
            q = np.conj(A) * B
            e = np.abs(A) ** 2
            rho = sm(q) / (sm(e) + 1e-3 * e.mean() + 1e-9)
            gain = np.conj(rho) / (np.abs(rho) ** 2 + 0.05)
            c1 = c1 * gain; cm = cm * np.conj(gain)
            if t == 0: rho_out = rho
        S1 = fft2(c1); Sm = fft2(cm)
        num += np.conj(H0) * Cw[0] + np.conj(w1 * H1) * S1 + np.conj(w1 * Hm) * Sm
        den += H0 ** 2 + (w1 * H1) ** 2 + (w1 * Hm) ** 2
    rec = np.real(ifft2(num / (den + lam) * APOD))
    return (rec, rho_out) if aux else rec

def widefield_wiener(raw, lam):
    return np.real(ifft2(fft2(raw.mean((0, 1))) * H0 / (H0 ** 2 + lam) * APOD_WF))

def estimate_pattern(raw, search=1.6, step=0.1, b=B0):
    """Self-calibration: find each pattern's frequency by maximising the cross-correlation between the
    zero-order and first-order components, and its global phase offset from the argument of the peak."""
    Fr = fft2(raw); p_est = np.zeros((3, 2)); phi_est = np.zeros((3, 3)); n = np.arange(NPIX)
    for t in range(3):
        px, py = P_NOM[t]
        Cw = _unmix(Fr[t], PH_NOM)
        Mk = (H0 > 0.25) & (otf_of(KX + px, KY + py) > 0.25)
        A = ifft2(np.where(Mk, Cw[0] / np.maximum(H0, 1e-3), 0))
        q = np.conj(A) * ifft2(Cw[1])
        def dft(gx, gy):
            return np.exp(-2j * np.pi * np.outer(gy, n)) @ q @ np.exp(-2j * np.pi * np.outer(gx, n)).T
        g = np.arange(-search, search + 1e-9, step) / NPIX
        Phi = dft(px + g, py + g); iy, ix = np.unravel_index(np.argmax(np.abs(Phi)), Phi.shape)
        cx, cy = px + g[ix], py + g[iy]
        fine = np.arange(-step, step + 1e-9, step / 10) / NPIX
        Phi = dft(cx + fine, cy + fine); iy, ix = np.unravel_index(np.argmax(np.abs(Phi)), Phi.shape)
        cx, cy = cx + fine[ix], cy + fine[iy]
        p_est[t] = (cx, cy); phi_est[t] = PH_NOM + np.angle(dft([cx], [cy])[0, 0])
    return p_est, phi_est

# ------------------------------------------------------------ metrics
BORDER = 16
_sl = (slice(BORDER, -BORDER), slice(BORDER, -BORDER))
def fit_to(rec, gt):
    r = rec[_sl].ravel(); g = gt[_sl].ravel()
    a, b = np.linalg.lstsq(np.stack([r, np.ones_like(r)], 1), g, rcond=None)[0]
    return np.clip(a * rec + b, 0, 1)
def score(rec, gt):
    r = fit_to(rec, gt)
    mse = np.mean((r[_sl] - gt[_sl]) ** 2)
    return 10 * np.log10(1 / (mse + 1e-12)), ssim_fn(gt[_sl], r[_sl], data_range=1.0)

# ------------------------------------------------------------ noise-aware regularisation for the baselines
_VAL_IMG = VAL[0] if len(VAL) else TRAIN[0]
LAM_GRID = np.logspace(-5, 0, 11)
def _tune(peaks):
    truth = make_truth(); best_s, best_w = [], []
    for pk in peaks:
        raw = simulate(_VAL_IMG, truth, pk, noise_seed=11)
        s = [score(reconstruct(raw, P_NOM, PH_ASSUMED, l), _VAL_IMG)[0] for l in LAM_GRID]
        w = [score(widefield_wiener(raw, l), _VAL_IMG)[0] for l in LAM_GRID]
        best_s.append(LAM_GRID[int(np.argmax(s))]); best_w.append(LAM_GRID[int(np.argmax(w))])
    return np.array(best_s), np.array(best_w)
_PK = np.array([3., 10., 30., 100., 300., 1000.])
_LS, _LW = _tune(_PK)
LAM    = lambda pk: float(np.exp(np.interp(np.log(pk), np.log(_PK), np.log(_LS))))
LAM_WF = lambda pk: float(np.exp(np.interp(np.log(pk), np.log(_PK), np.log(_LW))))
print("tuned Wiener lambda (SIM):", {int(k): float(round(v, 5)) for k, v in zip(_PK, _LS)}, "| (widefield):", {int(k): float(round(v, 5)) for k, v in zip(_PK, _LW)})

def run(gt, peak=100.0, phase_err=0.0, freq_err=0.0, eta=0.0, ofb=0.0, seed=0, mode="nominal", local=False):
    """simulate + reconstruct one condition. mode: nominal (design values) | oracle (true values) | auto (self-calibrated)."""
    truth = make_truth(phase_err, freq_err, eta, seed)
    raw = simulate(gt, truth, peak, ofb, noise_seed=5000 + seed)
    if mode == "nominal": p, phi = P_NOM, PH_ASSUMED
    elif mode == "oracle": p, phi = truth["p"], truth["phi"]
    else: p, phi = estimate_pattern(raw)
    rec = reconstruct(raw, p, phi, LAM(peak), local=local)
    return dict(rec=rec, raw=raw, truth=truth, p=p, phi=phi)

# ------------------------------------------------------------ sanity check
_t = TEST["Membranes"]; _o = run(_t, 1000, mode="oracle")
_wf = score(widefield_wiener(_o["raw"], LAM_WF(1000)), _t); _si = score(_o["rec"], _t)
print(f"sanity (1000 photons, perfect calibration): widefield+Wiener {_wf[0]:.2f} dB / SSIM {_wf[1]:.3f}  ->  SIM {_si[0]:.2f} dB / SSIM {_si[1]:.3f}")
assert _si[0] > _wf[0], "SIM reconstruction should beat widefield deconvolution under ideal conditions - engine bug"
log("cell 3 done")


widefield cut-off 0.2 cyc/px | pattern 0.170 | SIM support 0.370 -> 1.85x | pattern period 5.9 px
tuned Wiener lambda (SIM): {3: 1.0, 10: 0.31623, 30: 0.1, 100: 0.03162, 300: 0.01, 1000: 0.00316} | (widefield): {3: 0.1, 10: 0.01, 30: 0.00316, 100: 0.001, 300: 0.00032, 1000: 0.0001}
sanity (1000 photons, perfect calibration): widefield+Wiener 27.74 dB / SSIM 0.850  ->  SIM 32.23 dB / SSIM 0.924
[    13s] cell 3 done


In [5]:
# Cell 4 — Research questions 1 & 2: calibration sensitivity, self-calibration, uneven illumination
R = {}                                         # all results that end up in the dashboard
IM = list(TEST.items())                        # held-out images only
SEEDS = 1 if QUICK_MODE else 2

def run_multi(gt, peak, pe, fe, eta=0.0, ofb=0.0, seed=0, modes=("nominal", "auto", "oracle"), local=False):
    """Simulate once, reconstruct with several calibration strategies (same raw data for all)."""
    truth = make_truth(pe, fe, eta, seed)
    raw = simulate(gt, truth, peak, ofb, noise_seed=5000 + seed)
    out = dict(raw=raw, truth=truth); lam = LAM(peak)
    for m in modes:
        if m == "nominal": p, phi = P_NOM, PH_ASSUMED
        elif m == "oracle": p, phi = truth["p"], truth["phi"]
        else: p, phi = estimate_pattern(raw)
        out[m] = dict(rec=reconstruct(raw, p, phi, lam, local=local), p=p, phi=phi)
    return out

# ============ Q1a  PSNR landscape: pattern phase error x pattern frequency error (nominal calibration assumed)
PH_LIST = [0, 10, 20, 30, 45] if QUICK_MODE else [0, 5, 10, 20, 30, 45, 60]
FQ_LIST = [0, 0.15, 0.3, 0.5, 1.0] if QUICK_MODE else [0, 0.1, 0.2, 0.35, 0.5, 0.75, 1.0]
grid = np.zeros((len(PH_LIST), len(FQ_LIST)))
for a, pe in enumerate(PH_LIST):
    for b, fe in enumerate(FQ_LIST):
        v = [score(run_multi(gt, 100, pe, fe, seed=s, modes=("nominal",))["nominal"]["rec"], gt)[0] for _, gt in IM for s in range(SEEDS)]
        grid[a, b] = np.mean(v)
R.update(q1_grid=grid, q1_ph=PH_LIST, q1_fq=FQ_LIST)
_wf = [score(widefield_wiener(run_multi(gt, 100, 0, 0, seed=s, modes=())["raw"], LAM_WF(100)), gt)[0] for _, gt in IM for s in range(SEEDS)]
R["wf100"] = float(np.mean(_wf))
log(f"Q1a done | ideal {grid[0,0]:.2f} dB | worst {grid.min():.2f} dB | widefield+Wiener {R['wf100']:.2f} dB")

# residual spectra: where does the error energy sit in Fourier space?
_g = TEST["Membranes"]
_ok = run_multi(_g, 300, 0, 0, modes=("oracle",))["oracle"]["rec"]
_bad = run_multi(_g, 300, 20, 0.6, modes=("nominal",))["nominal"]["rec"]
R["spec_ideal"] = np.round(log_spectrum2d(fit_to(_ok, _g) - _g), 1)
R["spec_bad"]   = np.round(log_spectrum2d(fit_to(_bad, _g) - _g), 1)
R["spec_psnr"]  = (score(_ok, _g)[0], score(_bad, _g)[0])

# ============ Q1b  can the data calibrate itself?  (true errors: 10 deg phase, 0.8 cycles/FOV frequency)
PEAKS = [3, 8, 20, 50, 150, 500] if QUICK_MODE else [2, 4, 8, 16, 40, 100, 300, 1000]
E2 = []
for pk in PEAKS:
    for nm, gt in IM:
        for s in range(SEEDS + 1):
            o = run_multi(gt, pk, 10, 0.8, seed=s)
            perr = np.linalg.norm(o["auto"]["p"] - o["truth"]["p"], axis=1) * NPIX
            off_true = np.mean(o["truth"]["phi"] - PH_NOM[None, :], 1); off_est = o["auto"]["phi"][:, 0] - PH_NOM[0]
            d = np.angle(np.exp(1j * (off_est - off_true)))
            E2.append(dict(peak=pk, image=nm, seed=s, freq_err=perr.mean(), phase_err=np.degrees(np.abs(d)).mean(),
                           psnr_nominal=score(o["nominal"]["rec"], gt)[0], psnr_auto=score(o["auto"]["rec"], gt)[0],
                           psnr_oracle=score(o["oracle"]["rec"], gt)[0],
                           psnr_wf=score(widefield_wiener(o["raw"], LAM_WF(pk)), gt)[0]))
E2 = pd.DataFrame(E2); R["q1_est"] = E2
log("Q1b done\n" + E2.groupby("peak")[["freq_err", "phase_err", "psnr_nominal", "psnr_auto", "psnr_oracle", "psnr_wf"]].median().round(2).to_string())

# ============ Q2  spatially varying illumination: global-sinusoid model vs local pattern estimation (true calibration known)
ETA = [0, 0.5, 1.0] if QUICK_MODE else [0, 0.2, 0.4, 0.6, 0.8, 1.0]
E3 = []
for eta in ETA:
    for nm, gt in IM:
        for s in range(SEEDS + 1):
            for loc in (False, True):
                o = run_multi(gt, 100, 0, 0, eta=eta, seed=s, modes=("oracle",), local=loc)
                E3.append(dict(eta=eta, image=nm, seed=s, method="local" if loc else "global", psnr=score(o["oracle"]["rec"], gt)[0],
                               ssim=score(o["oracle"]["rec"], gt)[1]))
E3 = pd.DataFrame(E3); R["q2"] = E3
log("Q2 done\n" + E3.groupby(["eta", "method"]).psnr.mean().unstack().round(2).to_string())

# maps for the figure (strongest distortion, membranes)
_truth = make_truth(0, 0, 1.0, 0); _raw = simulate(_g, _truth, 100, 0, 5000)
_rg = reconstruct(_raw, _truth["p"], _truth["phi"], LAM(100))
_rl, _rho = reconstruct(_raw, _truth["p"], _truth["phi"], LAM(100), local=True, aux=True)
_psi_t = _truth["eta"] * 1.2 * _truth["fields"][0][0]
_psi_e = np.angle(_rho * np.exp(-1j * np.angle(np.sum(_rho * np.exp(-1j * _psi_t)))))      # remove the arbitrary global offset
R["q2_maps"] = dict(psi_true=np.round(_psi_t[::2, ::2], 3), psi_est=np.round(_psi_e[::2, ::2], 3),
                    err_global=np.round(np.abs(fit_to(_rg, _g) - _g)[::2, ::2], 3), err_local=np.round(np.abs(fit_to(_rl, _g) - _g)[::2, ::2], 3),
                    psnr=(score(_rg, _g)[0], score(_rl, _g)[0]),
                    psi_corr=float(np.corrcoef(_psi_t[BORDER:-BORDER, BORDER:-BORDER].ravel(), _psi_e[BORDER:-BORDER, BORDER:-BORDER].ravel())[0, 1]))
log(f"cell 4 done | phase-map correlation (true vs estimated) = {R['q2_maps']['psi_corr']:.2f}")


[    90s] Q1a done | ideal 29.65 dB | worst 25.65 dB | widefield+Wiener 27.25 dB
[   126s] Q1b done
      freq_err  phase_err  psnr_nominal  psnr_auto  psnr_oracle  psnr_wf
peak                                                                    
2         0.13      21.46         23.06      23.83        23.81    24.01
4         0.06      13.53         24.40      25.31        25.31    25.23
8         0.04       6.25         25.08      26.41        26.40    26.08
16        0.03       4.19         25.43      27.52        27.52    26.59
40        0.03       3.88         25.65      28.94        28.95    27.12
100       0.03       4.12         25.71      30.11        30.16    27.38
300       0.02       3.18         25.62      31.34        31.39    27.61
1000      0.02       3.02         25.56      32.20        32.28    27.73
[   156s] Q2 done
method  global  local
eta                  
0.0      29.66  29.58
0.2      29.46  29.50
0.4      29.24  29.45
0.6      28.95  29.36
0.8      28.64  29.2

In [6]:
# Cell 5 — Research question 3: a small learned module on top of the classical reconstruction
# Train on kidney + lily images, test on cells3d + HeLa images the network has never seen. Also: stress tests + data for the lab.
N_TRAIN = 60 if QUICK_MODE else 700
ITERS   = 120 if QUICK_MODE else 3000
BATCH, PATCH = 16, 128
rng = np.random.RandomState(SEED)

def draw_condition():
    return dict(peak=float(np.exp(rng.uniform(np.log(8), np.log(600)))), pe=rng.uniform(0, 30), fe=rng.uniform(0, 1.0),
                eta=rng.uniform(0, 0.8), ofb=rng.uniform(0, 0.2))
def make_set(imgs, n, tag):
    X = np.empty((n, NPIX, NPIX), np.float32); Y = np.empty_like(X); t0 = time.time()
    for i in range(n):
        gt = np.rot90(imgs[rng.randint(len(imgs))], rng.randint(4)); gt = np.ascontiguousarray(gt[:, ::-1] if rng.rand() < .5 else gt)
        c = draw_condition()
        X[i] = run(gt, c["peak"], c["pe"], c["fe"], c["eta"], c["ofb"], seed=int(rng.randint(0, 5000)), mode="nominal")["rec"]; Y[i] = gt
        if (i + 1) % 100 == 0: log(f"  {tag}: {i+1}/{n} simulated ({(time.time()-t0)/(i+1):.2f}s each)")
    return X, Y
Xtr, Ytr = make_set(TRAIN, N_TRAIN, "train"); Xva, Yva = make_set(VAL, 8 if QUICK_MODE else 40, "val")
log(f"training set {Xtr.shape} (peak 8-600 photons, phase err 0-30 deg, freq err 0-1 cyc/FOV, inhomogeneity 0-0.8, haze 0-0.2)")

class DC(nn.Module):
    def __init__(s, i, o):
        super().__init__(); s.f = nn.Sequential(nn.Conv2d(i, o, 3, padding=1), nn.GELU(), nn.Conv2d(o, o, 3, padding=1), nn.GELU())
    def forward(s, x): return s.f(x)
class UNet(nn.Module):                       # small residual U-Net: output = input + learned correction
    def __init__(s, c=32):
        super().__init__(); s.e1 = DC(1, c); s.e2 = DC(c, 2 * c); s.e3 = DC(2 * c, 4 * c)
        s.d2 = DC(6 * c, 2 * c); s.d1 = DC(3 * c, c); s.out = nn.Conv2d(c, 1, 1)
    def forward(s, x):
        e1 = s.e1(x); e2 = s.e2(F.avg_pool2d(e1, 2)); e3 = s.e3(F.avg_pool2d(e2, 2))
        up = lambda z: F.interpolate(z, scale_factor=2, mode="bilinear", align_corners=False)
        d2 = s.d2(torch.cat([up(e3), e2], 1)); d1 = s.d1(torch.cat([up(d2), e1], 1))
        return x + s.out(d1)
net = UNet().to(DEV); N_PARAMS = sum(p.numel() for p in net.parameters())
opt = torch.optim.AdamW(net.parameters(), lr=2e-3, weight_decay=1e-4)
sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=ITERS, eta_min=1e-5)
tX, tY = torch.tensor(Xtr)[:, None].to(DEV), torch.tensor(Ytr)[:, None].to(DEV)
vX, vY = torch.tensor(Xva)[:, None].to(DEV), torch.tensor(Yva)[:, None].to(DEV)

def batch():
    idx = torch.randint(0, tX.shape[0], (BATCH,)); xs, ys = [], []
    for i in idx.tolist():
        y0, x0 = np.random.randint(0, NPIX - PATCH + 1, 2)
        xs.append(tX[i, :, y0:y0 + PATCH, x0:x0 + PATCH]); ys.append(tY[i, :, y0:y0 + PATCH, x0:x0 + PATCH])
    x, y = torch.stack(xs), torch.stack(ys); k = np.random.randint(4)
    x, y = torch.rot90(x, k, (2, 3)), torch.rot90(y, k, (2, 3))
    if np.random.rand() < .5: x, y = x.flip(3), y.flip(3)
    return x, y

hist = dict(it=[], train=[], val_it=[], val=[]); run_loss = []; t0 = time.time()
with torch.no_grad(): v0 = float(F.l1_loss(net(vX), vY))
hist["val_it"].append(0); hist["val"].append(v0)
for it in range(1, ITERS + 1):
    x, y = batch(); loss = F.l1_loss(net(x), y)
    opt.zero_grad(set_to_none=True); loss.backward(); opt.step(); sched.step(); run_loss.append(loss.item())
    if it % 25 == 0: hist["it"].append(it); hist["train"].append(float(np.mean(run_loss[-25:])))
    if it % 100 == 0 or it == ITERS:
        with torch.no_grad(): hist["val_it"].append(it); hist["val"].append(float(F.l1_loss(net(vX), vY)))
        if it % 500 == 0 or it == ITERS: log(f"  iter {it}/{ITERS} | train L1 {hist['train'][-1]:.4f} | val L1 {hist['val'][-1]:.4f}")
net.eval(); TRAIN_SECONDS = time.time() - t0
with torch.no_grad(): v_in = float(F.l1_loss(vX, vY))
R["train"] = dict(hist=hist, params=N_PARAMS, seconds=TRAIN_SECONDS, n_train=N_TRAIN, iters=ITERS, val_input_l1=v_in, device=DEV,
                  n_images=len(TRAIN))
log(f"training done in {TRAIN_SECONDS:.0f}s | {N_PARAMS/1e3:.0f}k parameters | val L1: input {v_in:.4f} -> output {hist['val'][-1]:.4f}")

@torch.no_grad()
def apply_net(rec):
    return net(torch.tensor(np.ascontiguousarray(rec, dtype=np.float32))[None, None].to(DEV))[0, 0].cpu().numpy()

METHODS = ["Widefield + Wiener", "Classical SIM (design values)", "Self-calibrated SIM", "Self-calibrated + local pattern", "Classical + learned module", "Self-calibrated + local + learned"]
def all_methods(gt, peak, pe, fe, eta, ofb, seed):
    truth = make_truth(pe, fe, eta, seed); raw = simulate(gt, truth, peak, ofb, 5000 + seed); lam = LAM(peak)
    pa, pha = estimate_pattern(raw); cl = reconstruct(raw, P_NOM, PH_ASSUMED, lam)
    loc = reconstruct(raw, pa, pha, lam, local=True)
    return {METHODS[0]: widefield_wiener(raw, LAM_WF(peak)), METHODS[1]: cl, METHODS[2]: reconstruct(raw, pa, pha, lam),
            METHODS[3]: loc, METHODS[4]: apply_net(cl), METHODS[5]: apply_net(loc)}, raw

# ============ held-out evaluation at a realistic calibration condition
TYP = dict(pe=10, fe=0.4, eta=0.4, ofb=0.1)
EV_PEAKS = [10, 50, 300] if QUICK_MODE else [8, 20, 50, 150, 500]
Q3 = []; keep = {}
for pk in EV_PEAKS:
    for nm, gt in IM:
        for s in range(2 if QUICK_MODE else 3):
            recs, raw = all_methods(gt, pk, seed=100 + s, **TYP)
            for m, r in recs.items():
                p_, s_ = score(r, gt); Q3.append(dict(peak=pk, image=nm, seed=s, method=m, psnr=p_, ssim=s_))
            if nm == "Membranes" and s == 0: keep[pk] = (recs, gt)
Q3 = pd.DataFrame(Q3); R["q3"] = Q3
log("Q3 held-out evaluation done\n" + Q3.groupby(["peak", "method"]).psnr.mean().unstack()[METHODS].round(2).to_string())
_pk = EV_PEAKS[len(EV_PEAKS) // 2]; _recs, _gt = keep[_pk]
R["q3_spec"] = dict(peak=_pk, gt=radial_spectrum(_gt), **{m: radial_spectrum(fit_to(r, _gt)) for m, r in _recs.items()})
def _h2(r): return np.log10(np.histogram2d(_gt[_sl].ravel(), fit_to(r, _gt)[_sl].ravel(), bins=48, range=[[0, 1], [0, 1]])[0] + 1)
R["q3_hist"] = dict(classical=_h2(_recs[METHODS[1]]), learned=_h2(_recs[METHODS[4]]))
R["q3_examples"] = {m: img_uri(fit_to(r, _gt)) for m, r in _recs.items()}; R["q3_examples"]["Ground truth"] = img_uri(_gt)

# ============ stress tests
HAZE = [0, 0.2, 0.4] if QUICK_MODE else [0, 0.1, 0.2, 0.3, 0.4]
ST = []
for hz in HAZE:
    for nm, gt in IM:
        recs, _ = all_methods(gt, 50, 10, 0.4, 0.4, hz, 300)
        for m in (METHODS[1], METHODS[3], METHODS[4], METHODS[5]): ST.append(dict(test="haze", x=hz, image=nm, method=m, psnr=score(recs[m], gt)[0]))
for lab_, c in {"inside training range": (20, 0.6, 0.6), "beyond training range": (45, 1.5, 1.0)}.items():
    for nm, gt in IM:
        recs, _ = all_methods(gt, 50, c[0], c[1], c[2], 0.1, 400)
        for m in (METHODS[1], METHODS[3], METHODS[4], METHODS[5]): ST.append(dict(test="range", x=lab_, image=nm, method=m, psnr=score(recs[m], gt)[0]))
R["stress"] = pd.DataFrame(ST)
log("stress tests done\n" + R["stress"].groupby(["test", "x", "method"]).psnr.mean().unstack().round(2).to_string())

# ============ data for the interactive lab (membranes, 128x128 region of interest)
LAB_PEAKS = [100, 15] if QUICK_MODE else [400, 100, 40, 15]
LAB_PH    = [0, 30] if QUICK_MODE else [0, 10, 20, 30, 45]
LAB_FQ    = [0, 0.6] if QUICK_MODE else [0, 0.15, 0.3, 0.6, 1.0]
_g = TEST["Membranes"]
_best, _roi = -1, (64, 64)
for y0 in range(BORDER, NPIX - BORDER - 128 + 1, 16):
    for x0 in range(BORDER, NPIX - BORDER - 128 + 1, 16):
        sd = _g[y0:y0 + 128, x0:x0 + 128].std()
        if sd > _best: _best, _roi = sd, (y0, x0)
ry, rx = _roi; crop = lambda a: a[ry:ry + 128, rx:rx + 128]
LAB = dict(peaks=LAB_PEAKS, ph=LAB_PH, fq=LAB_FQ, gt=img_uri(crop(_g)), wf=[], wf_m=[], cl=[], loc=[], nn=[], m=[], par=[])
for pk in LAB_PEAKS:
    for pe in LAB_PH:
        for fe in LAB_FQ:
            truth = make_truth(pe, fe, 0.4, 0); raw = simulate(_g, truth, pk, 0.1, 5000); lam = LAM(pk)
            cl = reconstruct(raw, P_NOM, PH_ASSUMED, lam); lo = reconstruct(raw, P_NOM, PH_ASSUMED, lam, local=True); nn_ = apply_net(cl)
            if pe == LAB_PH[0] and fe == LAB_FQ[0]:
                w = raw.mean((0, 1)); LAB["wf"].append(img_uri(crop(fit_to(w, _g)))); LAB["wf_m"].append([round(float(v), 4) for v in score(w, _g)])
            LAB["cl"].append(img_uri(crop(fit_to(cl, _g)))); LAB["loc"].append(img_uri(crop(fit_to(lo, _g)))); LAB["nn"].append(img_uri(crop(fit_to(nn_, _g))))
            s_c, s_l, s_n = score(cl, _g), score(lo, _g), score(nn_, _g)
            LAB["m"].append([round(float(v), 4) for v in (*s_c, *s_l, *s_n)]); LAB["par"].append([float(pk), float(pe), float(fe), float(s_c[0]), float(s_l[0]), float(s_n[0])])
R["lab"] = LAB
log(f"cell 5 done | lab: {len(LAB['cl'])} conditions x 3 methods")


[   190s]   train: 100/700 simulated (0.33s each)
[   213s]   train: 200/700 simulated (0.28s each)
[   238s]   train: 300/700 simulated (0.27s each)
[   261s]   train: 400/700 simulated (0.26s each)
[   285s]   train: 500/700 simulated (0.26s each)
[   309s]   train: 600/700 simulated (0.25s each)
[   332s]   train: 700/700 simulated (0.25s each)
[   342s] training set (700, 256, 256) (peak 8-600 photons, phase err 0-30 deg, freq err 0-1 cyc/FOV, inhomogeneity 0-0.8, haze 0-0.2)
[   384s]   iter 500/3000 | train L1 0.0360 | val L1 0.0324
[   418s]   iter 1000/3000 | train L1 0.0337 | val L1 0.0295
[   454s]   iter 1500/3000 | train L1 0.0319 | val L1 0.0278
[   490s]   iter 2000/3000 | train L1 0.0309 | val L1 0.0270
[   527s]   iter 2500/3000 | train L1 0.0307 | val L1 0.0267
[   563s]   iter 3000/3000 | train L1 0.0308 | val L1 0.0266
[   563s] training done in 216s | 471k parameters | val L1: input 0.0355 -> output 0.0266
[   591s] Q3 held-out evaluation done
method  Widefield + Wi

In [7]:
# Cell 6 — Dashboard figures: every Plotly chart plus the numbers that feed the written findings
sgn  = lambda x, d=1: f"{'+' if x >= 0 else '−'}{abs(x):.{d}f}"
rgba = lambda h, a: f"rgba({int(h[1:3],16)},{int(h[3:5],16)},{int(h[5:7],16)},{a})"
METH_COL = {METHODS[0]: C["l"], METHODS[1]: C["d"], METHODS[2]: C["g3"], METHODS[3]: C["g2"], METHODS[4]: C["y"], METHODS[5]: mix(C["y"], C["d"], 0.45)}
F_ = {}                                                    # figure html blocks

# ------------------------------------------------------------ numbers used in the text
g_ = R["q1_grid"]; PHL, FQL = R["q1_ph"], R["q1_fq"]; E2 = R["q1_est"]; E3 = R["q2"]; Q3 = R["q3"]; LAB = R["lab"]
def first_drop(vals, xs, d):
    for i in range(1, len(vals)):
        if vals[0] - vals[i] >= d:
            v0, v1 = vals[0] - vals[i - 1], vals[0] - vals[i]; return xs[i - 1] + (d - v0) / (v1 - v0) * (xs[i] - xs[i - 1])
    return None
ideal, wf100 = g_[0, 0], R["wf100"]
thr_f, thr_p = first_drop(g_[0, :], FQL, 1.0), first_drop(g_[:, 0], PHL, 1.0)
med = E2.groupby("peak").median(numeric_only=True); pk_ref = min(med.index, key=lambda v: abs(np.log(v / 100)))
auto_better = [int(p) for p in med.index if med.loc[p, "psnr_auto"] > med.loc[p, "psnr_wf"]]
q2m = E3.groupby(["eta", "method"]).psnr.mean().unstack(); q2s = E3.groupby(["eta", "method"]).psnr.std().unstack()
q3m = Q3.groupby(["peak", "method"]).psnr.mean().unstack()[METHODS]; q3all = Q3.groupby("method").psnr.mean()
pv = Q3.pivot_table(index=["peak", "image", "seed"], columns="method", values="psnr")
gain_cl = (pv[METHODS[4]] - pv[METHODS[1]]); gain_loc = (pv[METHODS[5]] - pv[METHODS[3]])
tr = R["train"]; val_gain = 100 * (tr["val_input_l1"] - tr["hist"]["val"][-1]) / tr["val_input_l1"]
K = dict(support=KMAX / FC, sim_gain=ideal - wf100, auto_err=med.loc[pk_ref, "freq_err"], learned=gain_loc.mean())

# ------------------------------------------------------------ DATA section figures
vis = INV.copy(); vis["area"] = vis["shape"].str.split("x").apply(lambda a: int(a[0]) * int(a[1]))
labels, parents, values, colors, hov = ["Public images"], [""], [0], ["#ffffff"], [""]
for ds in vis.dataset.unique():
    labels.append(ds); parents.append("Public images"); values.append(0); colors.append(mix(C["l"], "#ffffff", 0.75)); hov.append("")
role_col = {"training": C["g3"], "held-out test": C["y"], "-": mix(C["l"], "#ffffff", 0.35)}
for _, r in vis.iterrows():
    labels.append(f"{r.dataset}:{r.channel}"); parents.append(r.dataset); values.append(r.area); colors.append(role_col[r.role])
    hov.append(f"{r.role}<br>{r['shape']} · SNR proxy {r.snr:.1f}<br>spectrum reaches {r.cutoff:.2f} cyc/px<br>{r.reason}")
fig = go.Figure(go.Treemap(labels=labels, parents=parents, values=values, marker=dict(colors=colors, line=dict(color="#ffffff", width=2)),
                           hovertext=hov, hoverinfo="text", textinfo="label", textfont=dict(color=C["d"], family=FONT), tiling=dict(pad=3)))
F_["treemap"] = fig_html(style(fig, 360, legend=False).update_layout(margin=dict(l=4, r=4, t=4, b=4)), "f_treemap")

ax = ["Spectral reach", "High-freq. energy", "SNR (log)", "Unsaturated", "Field size"]
def _radar(df):
    return [min(df.cutoff.mean() / 0.5, 1), df.hf.mean() / INV.hf.max(), np.log10(df.snr.mean()) / np.log10(INV.snr.max()),
            1 - min(df.sat.mean() / 0.02, 1), min(df.min_side.mean() / 922, 1)]
fig = go.Figure()
for ds, d in INV.groupby("dataset", sort=False):
    st = d.status.iloc[0]; role = d.role.iloc[0]
    col = {"training": C["g1"], "held-out test": C["y"]}.get(role, C["l"]); v = _radar(d[d.status == "kept"] if (d.status == "kept").any() else d)
    fig.add_trace(go.Scatterpolar(r=v + v[:1], theta=ax + ax[:1], name=f"{ds} ({role if role != '-' else 'excluded'})", fill="toself", fillcolor=rgba(col, 0.18),
                                  line=dict(color=col, width=2.2, dash="solid" if role != "-" else "dot"), hovertemplate="%{theta}: %{r:.2f}<extra>" + ds + "</extra>"))
fig.update_layout(polar=dict(radialaxis=dict(range=[0, 1], gridcolor=GRID, linecolor=C["l"], tickfont=dict(size=9)), angularaxis=dict(gridcolor=GRID, linecolor=C["l"]), bgcolor="rgba(0,0,0,0)"))
F_["radar"] = fig_html(style(fig, 360).update_layout(legend=dict(orientation="v", x=1.02, y=0.5, font=dict(size=11)), margin=dict(l=50, r=10, t=20, b=20)), "f_radar")

REP = {"Membranes": TEST["Membranes"], "Nuclei": TEST["Nuclei"], "HeLa mitosis": TEST["HeLa mitosis"],
       "Kidney": norm01(RAW["kidney"][8, :, :, 1]), "Lily": norm01(RAW["lily"][:, :, 0])}
REP_COL = [C["y"], mix(C["y"], C["d"], 0.35), C["l"], C["g3"], C["g2"]]
fig = go.Figure(); edges = np.linspace(0, 1, 49); xc = 0.5 * (edges[1:] + edges[:-1])
for i, (nm, im) in enumerate(REP.items()):
    h = np.histogram(im.ravel(), edges, density=True)[0] ** 0.5; h = h / h.max() * 1.6; y0 = i * 1.0
    fig.add_trace(go.Scatter(x=xc, y=[y0] * len(xc), mode="lines", line=dict(width=0), showlegend=False, hoverinfo="skip"))
    fig.add_trace(go.Scatter(x=xc, y=y0 + h, mode="lines", fill="tonexty", fillcolor=rgba(REP_COL[i], 0.55), line=dict(color=REP_COL[i], width=1.6), name=nm,
                             showlegend=False, hovertemplate=nm + "<br>intensity %{x:.2f}<extra></extra>"))
fig.update_yaxes(tickvals=list(range(len(REP))), ticktext=list(REP), showgrid=False, zeroline=False)
fig.update_xaxes(title="normalised intensity (0.5–99.8 percentile)")
F_["ridge"] = fig_html(style(fig, 360, legend=False).update_layout(margin=dict(l=100, r=20, t=10, b=54)), "f_ridge")

fig = go.Figure()
for i, (nm, im) in enumerate(REP.items()):
    fr, pw = radial_spectrum(im[:256, :256]); fig.add_trace(go.Scatter(x=fr, y=pw / pw[0], mode="lines", name=nm, line=dict(color=REP_COL[i], width=2.4)))
for xv, lb, col in [(FC, "widefield cut-off", C["d"]), (KMAX, "SIM support", C["g1"])]:
    fig.add_vline(x=xv, line=dict(color=col, dash="dash", width=1.5), annotation_text=lb, annotation_position="top", annotation_font=dict(color=col, size=11))
fig.update_yaxes(type="log", title="radial power (normalised)", exponentformat="power"); fig.update_xaxes(title="spatial frequency (cycles / pixel)", range=[0, 0.5])
F_["spectra"] = fig_html(style(fig, 360), "f_spectra")

# ------------------------------------------------------------ PRINCIPLE section: images + Fourier views
ry, rx = _roi; crop = lambda a: a[ry:ry + 128, rx:rx + 128]
CMP = {}
for nm, gt in IM:
    o = run_multi(gt, 200, 0, 0, modes=("oracle",))
    CMP[nm] = dict(wf=img_uri(fit_to(o["raw"].mean((0, 1)), gt), q=95), sim=img_uri(fit_to(o["oracle"]["rec"], gt), q=95), gt=img_uri(gt, q=95),
                   src=TEST_SRC[nm], psnr=[round(float(score(o["raw"].mean((0, 1)), gt)[0]), 2), round(float(score(o["oracle"]["rec"], gt)[0]), 2)])
    if nm == "Membranes": _o_mem = o
_truth = make_truth(0, 0, 0, 0); _raw = simulate(TEST["Membranes"], _truth, 100, 0, 5000)
RAWS = [[img_uri(crop(np.clip(_raw[t, j] / np.percentile(_raw, 99.8), 0, 1))) for j in range(3)] for t in range(3)]
def _crop_spec(z):
    f = np.fft.fftshift(np.fft.fftfreq(z.shape[0])); sel = np.abs(f) <= 0.4; return f[sel], z[np.ix_(sel, sel)]
fig = make_subplots(1, 3, subplot_titles=("one raw frame", "widefield + Wiener", "SIM reconstruction"), horizontal_spacing=0.07)
_specs = [log_spectrum2d(_raw[0, 0]), log_spectrum2d(widefield_wiener(_o_mem["raw"], LAM_WF(200))), log_spectrum2d(_o_mem["oracle"]["rec"])]
for i, z in enumerate(_specs):
    f, zc = _crop_spec(z); fig.add_trace(go.Heatmap(x=f, y=f, z=np.round(zc, 1), colorscale=SCALE_SPEC, showscale=False, zmin=float(np.percentile(zc, 15)), zmax=float(np.percentile(zc, 99.9)), hovertemplate="kx %{x:.2f} · ky %{y:.2f}<extra></extra>"), 1, i + 1)
    for rad, col, dash in [(FC, C["d"], "dash")] + ([(KMAX, C["g1"], "dash")] if i == 2 else []):
        fig.add_shape(type="circle", x0=-rad, x1=rad, y0=-rad, y1=rad, line=dict(color=col, width=2, dash=dash), row=1, col=i + 1)
for t in range(3):
    for s_ in (1, -1):
        fig.add_trace(go.Scatter(x=[s_ * P_NOM[t, 0]], y=[s_ * P_NOM[t, 1]], mode="markers", marker=dict(color=C["y"], size=8, line=dict(color=C["d"], width=1)),
                                 showlegend=False, hoverinfo="skip"), 1, 1)
for i in range(3): fig.update_xaxes(title="kx", scaleanchor=f"y{i+1 if i else ''}", row=1, col=i + 1); fig.update_yaxes(title="ky" if i == 0 else None, row=1, col=i + 1)
F_["fourier"] = fig_html(style(fig, 420, legend=False).update_layout(margin=dict(l=50, r=10, t=44, b=50)), "f_fourier")

# ------------------------------------------------------------ Q1 figures
_txt = [[f"{v:.1f}" + (" ▾" if v < wf100 else "") for v in row] for row in g_]
fig = go.Figure(go.Heatmap(x=[f"{v:g}" for v in FQL], y=[f"{v}°" for v in PHL], z=np.round(g_, 2), text=_txt, texttemplate="%{text}", colorscale=SCALE_Q, xgap=3, ygap=3,
                           textfont=dict(size=12, color=C["d"]), colorbar=dict(title="PSNR (dB)", thickness=12, len=0.9),
                           hovertemplate="freq err %{x} cyc/FOV · phase err %{y}<br>PSNR %{z:.2f} dB<extra></extra>"))
fig.update_xaxes(title="pattern-frequency error (cycles per field of view)", type="category"); fig.update_yaxes(title="pattern-phase error (° rms)", type="category")
F_["q1_heat"] = fig_html(style(fig, 430, legend=False).update_layout(margin=dict(l=62, r=10, t=14, b=56)), "f_q1_heat")

fig = go.Figure()
fig.add_trace(go.Surface(x=FQL, y=PHL, z=g_, colorscale=SCALE_Q, showscale=False, hovertemplate="freq %{x} · phase %{y}°<br>PSNR %{z:.2f} dB<extra></extra>",
                         contours=dict(z=dict(show=True, usecolormap=True, project_z=True))))
fig.add_trace(go.Surface(x=FQL, y=PHL, z=np.full_like(g_, wf100), colorscale=[[0, C["d"]], [1, C["d"]]], opacity=0.25, showscale=False, hoverinfo="skip"))
fig.update_layout(scene=dict(xaxis=dict(title="freq. error (cyc/FOV)", backgroundcolor="rgba(0,0,0,0)", gridcolor=GRID), yaxis=dict(title="phase error (°)", backgroundcolor="rgba(0,0,0,0)", gridcolor=GRID),
                             zaxis=dict(title="PSNR (dB)", backgroundcolor="rgba(0,0,0,0)", gridcolor=GRID), camera=dict(eye=dict(x=1.6, y=-1.5, z=0.9))))
F_["q1_surf"] = fig_html(style(fig, 430, legend=False).update_layout(margin=dict(l=0, r=0, t=0, b=0)), "f_q1_surf")

q1 = E2.groupby("peak"); lo, hi = q1.freq_err.quantile(0.25), q1.freq_err.quantile(0.75); mf = q1.freq_err.median(); mp = q1.phase_err.median()
fig = make_subplots(1, 2, specs=[[{"secondary_y": True}, {}]], subplot_titles=("calibration error vs. photon budget", "what it buys: PSNR vs. photon budget"), horizontal_spacing=0.2)
fig.add_trace(go.Scatter(x=mf.index, y=mf.values, mode="lines+markers", name="frequency error (cyc/FOV)", line=dict(color=C["g1"], width=3), marker=dict(size=9),
                         error_y=dict(type="data", symmetric=False, array=(hi - mf).values, arrayminus=(mf - lo).values, color=C["g1"], thickness=1.4)), 1, 1, secondary_y=False)
fig.add_trace(go.Scatter(x=mp.index, y=mp.values, mode="lines+markers", name="phase offset error (°)", line=dict(color=C["y"], width=3, dash="dot"), marker=dict(size=9, symbol="diamond", line=dict(color=C["d"], width=1))), 1, 1, secondary_y=True)
if thr_f: fig.add_hline(y=thr_f, line=dict(color=C["d"], dash="dash", width=1.5), annotation_text=f"1 dB damage level ≈ {thr_f:.2f} cyc/FOV", annotation_position="top left", annotation_font=dict(size=11), row=1, col=1)
pm = E2.groupby("peak")[["psnr_nominal", "psnr_auto", "psnr_oracle", "psnr_wf"]].mean()
for col_, nm_, c_, d_ in [("psnr_wf", "widefield + Wiener", C["l"], "solid"), ("psnr_nominal", "design values (no calibration)", C["d"], "solid"), ("psnr_auto", "self-calibrated", C["g1"], "solid"), ("psnr_oracle", "oracle (true values)", C["g2"], "dot")]:
    fig.add_trace(go.Scatter(x=pm.index, y=pm[col_], mode="lines+markers", name=nm_, line=dict(color=c_, width=3, dash=d_), marker=dict(size=8),
                             fill="tonexty" if col_ == "psnr_auto" else None, fillcolor=rgba(C["g1"], 0.14)), 1, 2)
fig.update_xaxes(type="log", title="peak photons per raw frame"); fig.update_yaxes(type="log", title="frequency error (cycles / FOV)", secondary_y=False, row=1, col=1)
fig.update_yaxes(title="phase offset error (°)", secondary_y=True, showgrid=False, row=1, col=1); fig.update_yaxes(title="PSNR (dB)", row=1, col=2)
F_["q1_est"] = fig_html(style(fig, 400).update_layout(legend=dict(orientation="h", y=-0.28, x=0, font=dict(size=11))), "f_q1_est")

fig = make_subplots(1, 2, subplot_titles=(f"calibrated & ideal · {R['spec_psnr'][0]:.1f} dB", f"20° phase / 0.6 cyc/FOV error · {R['spec_psnr'][1]:.1f} dB"), horizontal_spacing=0.07)
for i, z in enumerate([R["spec_ideal"], R["spec_bad"]]):
    f, zc = _crop_spec(z); fig.add_trace(go.Heatmap(x=f, y=f, z=zc, colorscale=SCALE_SPEC, zmin=min(R["spec_ideal"].min(), R["spec_bad"].min()) + 2, zmax=max(R["spec_ideal"].max(), R["spec_bad"].max()),
                                                 showscale=i == 1, colorbar=dict(title="log₁₀ power", thickness=10, len=0.85), hovertemplate="kx %{x:.2f} · ky %{y:.2f}<extra></extra>"), 1, i + 1)
    for t in range(3):
        for s_ in (1, -1):
            fig.add_trace(go.Scatter(x=[s_ * P_NOM[t, 0]], y=[s_ * P_NOM[t, 1]], mode="markers", marker=dict(color="rgba(0,0,0,0)", size=17, line=dict(color=C["y"], width=2)), showlegend=False,
                                     hovertemplate="pattern frequency<extra></extra>"), 1, i + 1)
for i in range(2): fig.update_xaxes(title="kx", scaleanchor=f"y{i+1 if i else ''}", row=1, col=i + 1)
fig.update_yaxes(title="ky", row=1, col=1)
F_["q1_res"] = fig_html(style(fig, 400, legend=False).update_layout(margin=dict(l=50, r=10, t=44, b=50)), "f_q1_res")

# ------------------------------------------------------------ Q2 figures
fig = go.Figure(); etas = list(q2m.index)
for meth, nm_, c_ in [("global", "global sinusoid model", C["d"]), ("local", "local pattern estimation", C["g1"])]:
    m, s = q2m[meth].values, q2s[meth].fillna(0).values
    fig.add_trace(go.Scatter(x=etas + etas[::-1], y=list(m + s) + list((m - s)[::-1]), fill="toself", fillcolor=rgba(c_, 0.13), line=dict(width=0), hoverinfo="skip", showlegend=False))
    fig.add_trace(go.Scatter(x=etas, y=m, mode="lines+markers", name=nm_, line=dict(color=c_, width=3.4), marker=dict(size=10, line=dict(color="#ffffff", width=2))))
fig.update_xaxes(title="distortion strength η  (0 = perfect sinusoid)"); fig.update_yaxes(title="PSNR (dB) — mean ± s.d. over images/seeds")
F_["q2_lines"] = fig_html(style(fig, 400), "f_q2_lines")

M = R["q2_maps"]; vmax = max(M["err_global"].max(), M["err_local"].max())
fig = make_subplots(2, 2, subplot_titles=("true pattern phase ψ (rad)", f"estimated ψ̂ (corr. {M['psi_corr']:.2f})", f"|error|, global model · {M['psnr'][0]:.1f} dB", f"|error|, local model · {M['psnr'][1]:.1f} dB"),
                    horizontal_spacing=0.04, vertical_spacing=0.11)
for i, (z, cs, zr) in enumerate([(M["psi_true"], SCALE_DIV, (-1.3, 1.3)), (M["psi_est"], SCALE_DIV, (-1.3, 1.3)), (M["err_global"], SCALE_SEQ, (0, vmax)), (M["err_local"], SCALE_SEQ, (0, vmax))]):
    r_, c_ = divmod(i, 2)
    fig.add_trace(go.Heatmap(z=z, colorscale=cs, zmin=zr[0], zmax=zr[1], showscale=False, hovertemplate="%{z:.2f}<extra></extra>"), r_ + 1, c_ + 1)
    fig.update_xaxes(visible=False, row=r_ + 1, col=c_ + 1); fig.update_yaxes(visible=False, autorange="reversed", scaleanchor=f"x{i+1 if i else ''}", row=r_ + 1, col=c_ + 1)
fig.update_annotations(font_size=12)
F_["q2_maps"] = fig_html(style(fig, 420, legend=False).update_layout(margin=dict(l=10, r=10, t=36, b=10)), "f_q2_maps")

# ------------------------------------------------------------ Q3 figures
h = tr["hist"]; fig = go.Figure()
fig.add_trace(go.Scatter(x=h["it"], y=h["train"], mode="lines", name="training L1", line=dict(color=C["g1"], width=3)))
fig.add_trace(go.Scatter(x=h["val_it"], y=h["val"], mode="lines+markers", name="validation L1 (unseen images)", line=dict(color=C["d"], width=2.4), marker=dict(size=7)))
fig.add_hline(y=tr["val_input_l1"], line=dict(color=C["y"], width=2.5, dash="dash"), annotation_text="classical reconstruction (network input)", annotation_position="top right", annotation_font=dict(size=11))
fig.update_xaxes(title="iteration"); fig.update_yaxes(title="L1 error to ground truth")
F_["q3_train"] = fig_html(style(fig, 360), "f_q3_train")

fig = go.Figure(); pks = sorted(Q3.peak.unique())
for m in METHODS:
    d = Q3[Q3.method == m].groupby("peak").psnr
    fig.add_trace(go.Bar(x=[f"{p} ph" for p in pks], y=d.mean().reindex(pks).values, name=m, marker=dict(color=METH_COL[m], line=dict(color="#ffffff", width=1)),
                         error_y=dict(type="data", array=d.std().reindex(pks).fillna(0).values, color=C["d"], thickness=1, width=3), hovertemplate=m + "<br>%{y:.2f} dB<extra></extra>"))
fig.update_layout(barmode="group", bargap=0.18); fig.update_xaxes(title="peak photons per raw frame"); fig.update_yaxes(title="PSNR (dB) on held-out images", range=[max(0, Q3.psnr.min() - 2), Q3.psnr.max() + 1.5])
F_["q3_bars"] = fig_html(style(fig, 430).update_layout(legend=dict(orientation="h", y=-0.26, x=0, font=dict(size=11))), "f_q3_bars")

dd = pv.reset_index(); dd["gain"] = dd[METHODS[5]] - dd[METHODS[3]]; dd = dd.groupby(["peak", "image"], as_index=False).mean(numeric_only=True).sort_values("gain")
dd["row"] = dd.image + " · " + dd.peak.astype(int).astype(str) + " ph"; fig = go.Figure()
for _, r in dd.iterrows():
    fig.add_trace(go.Scatter(x=[r[METHODS[3]], r[METHODS[5]]], y=[r.row] * 2, mode="lines", line=dict(color=C["g1"] if r.gain >= 0 else C["d"], width=3), showlegend=False, hoverinfo="skip"))
fig.add_trace(go.Scatter(x=dd[METHODS[3]], y=dd.row, mode="markers", name="self-calibrated + local (classical)", marker=dict(color=METH_COL[METHODS[3]], size=11, line=dict(color="#ffffff", width=2))))
fig.add_trace(go.Scatter(x=dd[METHODS[5]], y=dd.row, mode="markers", name="+ learned module", marker=dict(color=METH_COL[METHODS[5]], size=11, symbol="diamond", line=dict(color="#ffffff", width=2)),
                         customdata=dd.gain, hovertemplate="%{y}<br>%{x:.2f} dB (%{customdata:+.2f})<extra></extra>"))
fig.update_xaxes(title="PSNR (dB)"); fig.update_yaxes(showgrid=True, automargin=True)
F_["q3_dumb"] = fig_html(style(fig, 520).update_layout(margin=dict(l=150, r=20, t=40, b=54)), "f_q3_dumb")

S = R["q3_spec"]; fig = go.Figure(); fr, pw = S["gt"]; fig.add_trace(go.Scatter(x=fr, y=pw / pw[0], name="ground truth", line=dict(color=C["d"], width=4), opacity=0.35))
for m in METHODS[:5]:
    fr, pw = S[m]; fig.add_trace(go.Scatter(x=fr, y=pw / pw[0], name=m, line=dict(color=METH_COL[m], width=2.4)))
for xv, lb, col in [(FC, "widefield cut-off", C["d"]), (KMAX, "SIM support", C["g1"])]:
    fig.add_vline(x=xv, line=dict(color=col, dash="dash", width=1.4), annotation_text=lb, annotation_position="top", annotation_font=dict(color=col, size=11))
fig.update_yaxes(type="log", title="radial power (norm.)", exponentformat="power"); fig.update_xaxes(title="spatial frequency (cycles / pixel)", range=[0, 0.5])
F_["q3_spec"] = fig_html(style(fig, 400).update_layout(legend=dict(orientation="h", y=-0.3, x=0, font=dict(size=11))), "f_q3_spec")

fig = make_subplots(1, 2, subplot_titles=("classical (design values)", "classical + learned module"), horizontal_spacing=0.1)
for i, k in enumerate(["classical", "learned"]):
    fig.add_trace(go.Heatmap(z=R["q3_hist"][k].T, x=np.linspace(0, 1, 48), y=np.linspace(0, 1, 48), colorscale=SCALE_SEQ, showscale=False, hovertemplate="truth %{x:.2f} · recon %{y:.2f}<extra></extra>"), 1, i + 1)
    fig.add_trace(go.Scatter(x=[0, 1], y=[0, 1], mode="lines", line=dict(color=C["y"], dash="dash", width=2), showlegend=False, hoverinfo="skip"), 1, i + 1)
    fig.update_xaxes(title="ground-truth intensity", row=1, col=i + 1)
fig.update_yaxes(title="reconstructed intensity", row=1, col=1)
F_["q3_hist"] = fig_html(style(fig, 360, legend=False).update_layout(margin=dict(l=56, r=10, t=44, b=54)), "f_q3_hist")

st = R["stress"]; fig = make_subplots(1, 2, subplot_titles=("out-of-focus haze (network trained up to 0.2)", "calibration errors inside / beyond training range"), horizontal_spacing=0.1)
sm = [METHODS[1], METHODS[3], METHODS[4], METHODS[5]]
for m in sm:
    d = st[(st.test == "haze") & (st.method == m)].groupby("x").psnr.mean()
    fig.add_trace(go.Scatter(x=d.index.astype(float), y=d.values, mode="lines+markers", name=m, line=dict(color=METH_COL[m], width=3), marker=dict(size=8)), 1, 1)
    d = st[(st.test == "range") & (st.method == m)].groupby("x").psnr.mean().reindex(["inside training range", "beyond training range"])
    fig.add_trace(go.Bar(x=d.index, y=d.values, name=m, marker_color=METH_COL[m], showlegend=False, hovertemplate=m + "<br>%{y:.2f} dB<extra></extra>"), 1, 2)
fig.update_xaxes(title="haze fraction", row=1, col=1); fig.update_yaxes(title="PSNR (dB)", row=1, col=1); fig.update_layout(barmode="group", bargap=0.2)
fig.update_yaxes(range=[max(0, st.psnr.min() - 2), st.psnr.max() + 1], row=1, col=2)
F_["q3_stress"] = fig_html(style(fig, 400).update_layout(legend=dict(orientation="h", y=-0.28, x=0, font=dict(size=11))), "f_q3_stress")

P_ = np.array(LAB["par"]); gmax = max(0.1, np.abs(P_[:, 5] - P_[:, 3]).max())
fig = go.Figure(go.Parcoords(line=dict(color=P_[:, 5] - P_[:, 3], colorscale=[[0, C["d"]], [0.5, "#e9eaea"], [1, C["g1"]]], cmin=-gmax, cmax=gmax, showscale=True,
                                       colorbar=dict(title="Δ PSNR<br>(learned − classical)", thickness=12, len=0.8)),
                             dimensions=[dict(label="photons / frame", values=P_[:, 0], range=[P_[:, 0].min(), P_[:, 0].max()]),
                                         dict(label="phase error (°)", values=P_[:, 1], range=[0, max(1, P_[:, 1].max())]),
                                         dict(label="freq. error (cyc/FOV)", values=P_[:, 2], range=[0, max(0.1, P_[:, 2].max())]),
                                         dict(label="classical PSNR", values=P_[:, 3]), dict(label="local-pattern PSNR", values=P_[:, 4]), dict(label="learned PSNR", values=P_[:, 5])]))
F_["parcoords"] = fig_html(style(fig, 400, legend=False).update_layout(margin=dict(l=70, r=40, t=50, b=24)), "f_parcoords")
log("figures built")


[   649s] figures built


In [8]:
# Cell 7 — Assemble the single-file HTML dashboard and save it to Google Drive
verdict = lambda x: "adds value" if x > 0.15 else ("is roughly neutral" if x > -0.15 else "does not help")
pos_frac = 100 * (gain_loc > 0).mean(); gl = gain_loc.groupby("peak").mean()
oracle_gap = float((med.psnr_oracle - med.psnr_auto).abs().max())
st_ = R["stress"]; _sp = st_.groupby(["test", "x", "method"]).psnr.mean()
gain_in = _sp[("range", "inside training range", METHODS[5])] - _sp[("range", "inside training range", METHODS[3])]
gain_out = _sp[("range", "beyond training range", METHODS[5])] - _sp[("range", "beyond training range", METHODS[3])]
hz_hi = max(HAZE); gain_hz0 = _sp[("haze", 0, METHODS[5])] - _sp[("haze", 0, METHODS[3])]; gain_hz1 = _sp[("haze", hz_hi, METHODS[5])] - _sp[("haze", hz_hi, METHODS[3])]
n_kept = int((INV.status == "kept").sum()); n_exc = int((INV.status != "kept").sum())

T = {}
T["q1a"] = (f"Pattern <b>frequency</b> is the fragile parameter: an error of just <b>{thr_f:.2f} cycles per field of view</b> already costs 1 dB. " if thr_f else
            f"Over the tested range, frequency errors never cost more than 1 dB. ") + \
           (f"Pattern <b>phase</b> errors are forgiven far more easily: even {PHL[-1]}° rms costs only {g_[0,0]-g_[-1,0]:.1f} dB." if thr_p is None else f"Phase errors cost 1 dB at about {thr_p:.0f}° rms.")
T["q1b"] = (f"With wrong frequencies the SIM result falls to {g_.min():.1f} dB, {'below' if g_.min() < wf100 else 'but still above'} plain widefield + Wiener ({wf100:.1f} dB): "
            f"{'a badly calibrated SIM system is worse than no SIM at all.' if g_.min() < wf100 else 'calibration decides how much of the gain survives.'}")
T["q1c"] = (f"Self-calibration (cross-correlating the zero- and first-order components) finds the pattern frequency to within <b>{med.loc[pk_ref,'freq_err']:.2f} cycles/FOV</b> at ~{pk_ref} photons and recovers "
            f"<b>{sgn(med.loc[pk_ref,'psnr_auto']-med.loc[pk_ref,'psnr_nominal'])} dB</b>, staying within {oracle_gap:.2f} dB of the oracle that knows the true values. " +
            (f"Even at the lowest budget tested ({int(med.index.min())} photons) it beats widefield + Wiener." if len(auto_better) == len(med) else
             f"It beats widefield + Wiener from {min(auto_better) if auto_better else '—'} photons per frame upward."))
T["q2a"] = (f"A global sinusoid model loses <b>{q2m['global'].iloc[0]-q2m['global'].iloc[-1]:.1f} dB</b> as the distortion grows from η=0 to η={etas[-1]:g}; estimating the pattern locally limits the loss to "
            f"<b>{q2m['local'].iloc[0]-q2m['local'].iloc[-1]:.1f} dB</b> (a {sgn(q2m['local'].iloc[-1]-q2m['global'].iloc[-1])} dB difference at the strongest distortion). "
            f"The price: {sgn(q2m['local'].iloc[0]-q2m['global'].iloc[0], 2)} dB when the pattern is already perfect.")
T["q2b"] = f"The estimated phase map reproduces the hidden one (correlation <b>{M['psi_corr']:.2f}</b>) using only the raw frames — the same data a real acquisition provides."
T["q3a"] = (f"On unseen validation images the network cuts the L1 error from {tr['val_input_l1']:.4f} to {tr['hist']['val'][-1]:.4f} (<b>{val_gain:.0f}%</b> lower) "
            f"after {tr['iters']} iterations ({tr['seconds']/60:.1f} min on {'a GPU' if tr['device']=='cuda' else 'CPU'}, {tr['params']/1e3:.0f}k parameters).")
T["q3b"] = (f"On held-out images the module changes the design-value reconstruction by <b>{sgn(gain_cl.mean())} dB</b> and the best classical pipeline (self-calibrated + local) by "
            f"<b>{sgn(gain_loc.mean())} dB</b> — it {verdict(gain_loc.mean())} there, with a gain in {pos_frac:.0f}% of the {len(gain_loc)} test conditions. "
            f"The effect is largest at {int(gl.idxmax())} photons ({sgn(gl.max())} dB) and smallest at {int(gl.idxmin())} photons ({sgn(gl.min())} dB).")
T["q3c"] = (f"Stress tests: with calibration errors inside the training range the module changes the best classical result by {sgn(gain_in)} dB, beyond the range by {sgn(gain_out)} dB; "
            f"with out-of-focus haze it goes from {sgn(gain_hz0)} dB (no haze) to {sgn(gain_hz1)} dB (haze {hz_hi:g}).")

raw_html = "".join(f'<figure><img src="{RAWS[t][j]}" alt="raw frame"><figcaption>θ = {[0,60,120][t]}° · φ = {[0,120,240][j]}°</figcaption></figure>' for t in range(3) for j in range(3))
test_html = "".join(f'<figure><img src="{CMP[n]["gt"]}" alt="{n}"><figcaption><b>{n}</b><br>{CMP[n]["src"]}</figcaption></figure>' for n in CMP)
params = [("Field of view", "256 × 256 px"), ("Widefield cut-off", f"{FC} cycles/px (circular-pupil OTF)"), ("Illumination pattern", f"{P0:.3f} cycles/px · period {1/P0:.1f} px · modulation {B0}"),
          ("Acquisition", "3 orientations (0°, 60°, 120°) × 3 phases = 9 raw frames"), ("Noise", "Poisson (peak photons per frame, 2 → 1000) + Gaussian read noise (1 e⁻)"),
          ("Classical pipeline", "component unmixing → sub-pixel shift → generalised Wiener (λ tuned per photon level on a validation image) → apodisation"),
          ("Self-calibration", "cross-correlation peak search (±1.6 cyc/FOV, 0.01 refinement) + global phase offset per orientation"),
          ("Local pattern model", "Gaussian-smoothed (σ = 14 px) complex gain between zero- and first-order components, divided out"),
          ("Metrics", "PSNR / SSIM on the 224² interior; reconstruction intensity-matched to ground truth by least squares"),
          ("Learned module", f"residual U-Net, {tr['params']/1e3:.0f}k parameters, {tr['n_train']} simulated reconstructions from {tr['n_images']} kidney/lily crops, L1 loss, {tr['iters']} iterations"),
          ("Held-out test", "cells3d (membranes, nuclei) and HeLa mitosis crop — never used for tuning or training"), ("Seed", str(SEED))]
params_html = "".join(f"<tr><th>{a}</th><td>{b}</td></tr>" for a, b in params)

TEMPLATE = r'''<!doctype html>
<html lang="en"><head><meta charset="utf-8"><meta name="viewport" content="width=device-width,initial-scale=1">
<title>SIM Lab — calibration, illumination and learning in structured-illumination microscopy</title>
<meta name="description" content="Interactive mini-project on structured-illumination microscopy reconstruction using public fluorescence images.">
<style>
:root{--g1:#1dad22;--g2:#268c32;--g3:#37bf53;--y:#e0d01f;--d:#494b52;--l:#bdbebf;--line:#e7e8e8;--soft:#f5faf5;--ysoft:#fcf9de}
*{box-sizing:border-box}html{scroll-behavior:smooth;scroll-padding-top:70px}
body{margin:0;font-family:Inter,system-ui,-apple-system,"Segoe UI",Roboto,Helvetica,Arial,sans-serif;color:var(--d);background:#fff;line-height:1.62;font-size:16px}
a{color:var(--g2)}h1,h2,h3,h4{margin:0;line-height:1.18;color:#3b3d43}
.wrap{max-width:1180px;margin:0 auto;padding:0 24px}
nav{position:sticky;top:0;z-index:50;background:rgba(255,255,255,.88);backdrop-filter:blur(10px);border-bottom:1px solid var(--line)}
nav .wrap{display:flex;align-items:center;gap:26px;height:56px;overflow-x:auto;white-space:nowrap}
nav b{color:var(--g1);font-size:17px;letter-spacing:.2px}
nav a{color:var(--d);text-decoration:none;font-size:14px;padding:6px 1px;border-bottom:2px solid transparent}
nav a:hover,nav a.on{border-color:var(--g1);color:#2d2f34}
.hero{position:relative;overflow:hidden;padding:70px 0 54px;background:radial-gradient(800px 380px at 88% -8%,rgba(55,191,83,.20),transparent 60%),radial-gradient(520px 320px at -4% 105%,rgba(224,208,31,.17),transparent 60%)}
.hero .wrap{display:grid;grid-template-columns:1.15fr .85fr;gap:46px;align-items:center}
.eyebrow{display:inline-block;font-size:12.5px;letter-spacing:.14em;text-transform:uppercase;font-weight:700;color:var(--g2);background:rgba(55,191,83,.13);padding:6px 12px;border-radius:99px}
h1{font-size:clamp(44px,7vw,76px);letter-spacing:-.03em;margin:16px 0 8px}h1 span{color:var(--g1)}
.sub{font-size:clamp(18px,2.2vw,23px);color:var(--d);max-width:640px;margin:0 0 18px}
.lead{max-width:640px;color:#5b5d65}
.chips{display:flex;flex-wrap:wrap;gap:8px;margin-top:20px}.chip{font-size:13px;border:1px solid var(--line);background:#fff;border-radius:99px;padding:5px 12px}.chip i{display:inline-block;width:8px;height:8px;border-radius:50%;background:var(--y);margin-right:7px}
.diag{position:relative;aspect-ratio:1;border-radius:24px;overflow:hidden;box-shadow:0 22px 60px rgba(38,140,50,.20);border:1px solid var(--line);background:#000}
.diag img{position:absolute;inset:0;width:100%;height:100%;object-fit:cover}.diag .wf{clip-path:polygon(0 0,100% 0,0 100%)}
.diag svg{position:absolute;inset:0;width:100%;height:100%}
.tag{position:absolute;font-size:12px;font-weight:700;letter-spacing:.06em;text-transform:uppercase;background:rgba(255,255,255,.92);color:var(--d);border-radius:99px;padding:5px 11px}
.kpis{display:grid;grid-template-columns:repeat(4,1fr);gap:14px;margin-top:-26px;position:relative;z-index:2}
.kpi{background:#fff;border:1px solid var(--line);border-radius:18px;padding:18px 20px;box-shadow:0 10px 34px rgba(38,140,50,.08)}
.kpi b{display:block;font-size:34px;letter-spacing:-.02em;color:var(--g1);line-height:1.05}.kpi:nth-child(2) b{color:var(--g2)}.kpi:nth-child(3) b{color:#b3a60f}.kpi:nth-child(4) b{color:var(--d)}
.kpi span{font-size:13.5px;color:#666870;display:block;margin-top:6px}
.sec{padding:72px 0}.alt{background:var(--soft)}
.sh{display:flex;gap:16px;align-items:flex-start;margin-bottom:22px}
.bd{flex:none;width:42px;height:42px;border-radius:13px;background:var(--g1);color:#fff;font-weight:800;display:grid;place-items:center;font-size:18px}
.sh h2{font-size:clamp(26px,3.4vw,36px);letter-spacing:-.02em}.sh p{margin:8px 0 0;max-width:780px;color:#5b5d65}
.rq{display:inline-block;background:var(--y);color:#3b3d43;font-weight:800;font-size:11.5px;letter-spacing:.1em;padding:3px 9px;border-radius:7px;margin-bottom:8px}
.card{background:#fff;border:1px solid var(--line);border-radius:20px;padding:20px 22px;box-shadow:0 8px 28px rgba(38,140,50,.055)}
.card h3{font-size:17px;margin-bottom:2px}.cap{font-size:13.5px;color:#6a6c74;margin:0 0 6px}
.g2{display:grid;grid-template-columns:1fr 1fr;gap:20px;margin-bottom:20px}.g3{display:grid;grid-template-columns:repeat(3,1fr);gap:16px;margin-bottom:20px}
.find{border-left:4px solid var(--y);background:var(--ysoft);padding:12px 16px;border-radius:0 12px 12px 0;margin:12px 0;font-size:15px}
.find:before{content:"Finding";font-size:11px;font-weight:800;letter-spacing:.12em;text-transform:uppercase;color:#9b9010;display:block;margin-bottom:2px}
figure{margin:0}figure img{width:100%;border-radius:14px;border:1px solid var(--line);display:block}figcaption{font-size:12.5px;color:#6a6c74;margin-top:6px;line-height:1.4}
.strip{display:grid;grid-template-columns:repeat(3,1fr);gap:16px}.raws{display:grid;grid-template-columns:repeat(3,1fr);gap:10px}
.rules{display:grid;grid-template-columns:repeat(4,1fr);gap:12px;margin:0 0 20px}.rule{background:#fff;border:1px solid var(--line);border-radius:14px;padding:12px 14px;font-size:14px}.rule b{display:block;color:var(--g2);font-size:22px;line-height:1.1}
.ctl{display:grid;grid-template-columns:repeat(3,1fr);gap:26px}.ctl label{font-size:13px;font-weight:700;letter-spacing:.06em;text-transform:uppercase;color:#6a6c74;display:flex;justify-content:space-between}
.ctl output{color:var(--g2);text-transform:none;letter-spacing:0;font-size:14px}
input[type=range]{width:100%;accent-color:var(--g1);height:28px}
.pre{display:flex;gap:8px;flex-wrap:wrap;margin:14px 0 4px}.pre button,.seg button{font:inherit;font-size:13.5px;border:1px solid var(--line);background:#fff;color:var(--d);border-radius:99px;padding:6px 14px;cursor:pointer;transition:.15s}
.pre button:hover,.seg button:hover{border-color:var(--g1)}.seg button.on{background:var(--g1);color:#fff;border-color:var(--g1)}
.labgrid{display:grid;grid-template-columns:repeat(5,1fr);gap:12px;margin-top:16px}
.pn h4{font-size:13px;margin:0 0 6px;display:flex;align-items:center;gap:6px}.pn h4 i{width:9px;height:9px;border-radius:3px;display:inline-block}
.pn img{width:100%;aspect-ratio:1;border-radius:12px;border:1px solid var(--line);display:block;background:#000}
.pn .m{font-size:13px;margin-top:6px;font-variant-numeric:tabular-nums}.pn .m b{font-size:18px;color:#3b3d43}.dl{font-weight:700}
.cmp{position:relative;aspect-ratio:1;max-width:560px;margin:0 auto;overflow:hidden;border-radius:18px;border:1px solid var(--line);background:#000;user-select:none}
.cmp img{position:absolute;inset:0;width:100%;height:100%;object-fit:cover}.cmp .hd{position:absolute;top:0;bottom:0;width:3px;background:#fff;box-shadow:0 0 0 1px rgba(0,0,0,.25);transform:translateX(-1.5px);pointer-events:none}
.seg{display:flex;gap:8px;flex-wrap:wrap;margin:0 0 12px;justify-content:center}
table.p{border-collapse:collapse;width:100%;font-size:14px}table.p th{text-align:left;width:28%;color:#6a6c74;font-weight:600;padding:9px 12px 9px 0;vertical-align:top;border-bottom:1px solid var(--line)}table.p td{padding:9px 0;border-bottom:1px solid var(--line)}
ul.lim{margin:0;padding-left:20px}ul.lim li{margin:8px 0}
footer{padding:38px 0 60px;color:#6a6c74;font-size:14px;border-top:1px solid var(--line)}
.rv{opacity:0;transform:translateY(14px);transition:opacity .7s ease,transform .7s ease}.rv.in{opacity:1;transform:none}
.noanim .rv{transition:none!important}.raws9{display:grid;grid-template-columns:repeat(9,1fr);gap:8px}.mt{margin-top:20px}.small{font-size:13.5px;color:#6a6c74}
@media(max-width:1000px){.hero .wrap{grid-template-columns:1fr}.kpis{grid-template-columns:repeat(2,1fr)}.labgrid{grid-template-columns:repeat(3,1fr)}.g2{grid-template-columns:1fr}.rules{grid-template-columns:repeat(2,1fr)}}
@media(max-width:640px){.raws9{grid-template-columns:repeat(3,1fr)}.labgrid{grid-template-columns:repeat(2,1fr)}.ctl{grid-template-columns:1fr}.strip,.g3{grid-template-columns:1fr}.sec{padding:52px 0}.kpi b{font-size:28px}}
</style>
<script>@@PLOTLY@@</script></head>
<body>
<nav><div class="wrap"><b>SIM Lab</b><a href="#data">Data</a><a href="#principle">Principle</a><a href="#lab">Interactive lab</a><a href="#q1">Q1 · Calibration</a><a href="#q2">Q2 · Illumination</a><a href="#q3">Q3 · Learning</a><a href="#limits">Method &amp; limits</a></div></nav>

<header class="hero"><div class="wrap">
 <div>
  <span class="eyebrow">Computational imaging · mini-project</span>
  <h1>SIM <span>Lab</span></h1>
  <p class="sub">What breaks structured-illumination reconstruction — and what repairs it?</p>
  <p class="lead">A small, fully reproducible study on public fluorescence images. I simulate the SIM acquisition, reconstruct it with a classical Wiener pipeline, and ask three questions about <b>calibration</b>, <b>inhomogeneous illumination</b> and a <b>learned module</b> — the themes of the M2 internship <i>“Image Reconstruction for Structured-Illumination Microscopy”</i> (IRIT, Toulouse) that inspired it.</p>
  <div class="chips"><span class="chip"><i></i>@@AUTHOR@@</span><span class="chip">@@DATE@@</span><span class="chip">Google Colab · @@DEVICE@@ · @@RUNTIME@@ min end-to-end</span><span class="chip">public data only</span></div>
 </div>
 <div class="diag"><img src="@@HERO_SIM@@" alt="SIM reconstruction"><img class="wf" src="@@HERO_WF@@" alt="widefield">
  <svg viewBox="0 0 100 100" preserveAspectRatio="none"><line x1="100" y1="0" x2="0" y2="100" stroke="#fff" stroke-width=".7"/></svg>
  <span class="tag" style="left:14px;top:14px">Widefield</span><span class="tag" style="right:14px;bottom:14px">SIM reconstruction</span></div>
</div></header>

<div class="wrap"><div class="kpis rv">
 <div class="kpi"><b>×@@K1@@</b><span>Fourier support vs. widefield (theory: pattern at 0.85 of the cut-off)</span></div>
 <div class="kpi"><b>@@K2@@ dB</b><span>SIM vs. widefield + Wiener, ideal calibration, 100 photons</span></div>
 <div class="kpi"><b>@@K3@@</b><span>cycles/FOV · median self-calibration error at ~@@K3P@@ photons</span></div>
 <div class="kpi"><b>@@K4@@ dB</b><span>learned module on top of the best classical pipeline (mean, held-out images)</span></div>
</div></div>

<section class="sec" id="data"><div class="wrap">
 <div class="sh rv"><div class="bd">1</div><div><h2>Data: what the public images allow</h2><p>@@N_ALL@@ frames from six public datasets were screened with an exploration notebook. Four rules decide what enters the study; the split is by <i>dataset</i>, so the test images come from different samples and microscopes than the training images.</p></div></div>
 <div class="rules rv"><div class="rule"><b>≥ 256 px</b>field of view must fit the simulation</div><div class="rule"><b>≥ 0.35 cyc/px</b>spectrum must reach fine detail (SIM has something to recover)</div><div class="rule"><b>&lt; 1 %</b>saturated pixels</div><div class="rule"><b>@@N_KEPT@@ kept · @@N_EXC@@ excluded</b>train: kidney, lily · test: cells3d, HeLa</div></div>
 <div class="g2">
  <div class="card rv"><h3>Inventory</h3><p class="cap">Tile area = pixel count · green = training · yellow = held-out test · grey = excluded (hover for the reason)</p>@@F_TREEMAP@@</div>
  <div class="card rv"><h3>Dataset fingerprints</h3><p class="cap">Normalised properties of each dataset (kept channels averaged)</p>@@F_RADAR@@</div>
 </div>
 <div class="g2">
  <div class="card rv"><h3>Intensity distributions</h3><p class="cap">Ridgeline of the ground-truth images after percentile normalisation</p>@@F_RIDGE@@</div>
  <div class="card rv"><h3>Spatial-frequency content</h3><p class="cap">The ground truths carry detail well beyond the widefield cut-off — that is what SIM tries to bring back</p>@@F_SPECTRA@@</div>
 </div>
 <div class="card rv"><h3>Held-out ground truths used for every test</h3><p class="cap">Never seen during training or parameter tuning</p><div class="strip">@@TEST_HTML@@</div></div>
</div></section>

<section class="sec alt" id="principle"><div class="wrap">
 <div class="sh rv"><div class="bd">2</div><div><h2>Principle: how a striped pattern beats the diffraction limit</h2><p>A microscope only transmits spatial frequencies up to a cut-off (the circle below). Illuminating with a fine stripe pattern shifts high-frequency object content into that circle. Reconstruction puts the shifted copies back where they belong — enlarging the usable Fourier disc.</p></div></div>
 <div class="g2">
  <div class="card rv"><h3>Fourier-space explorer</h3><p class="cap">Move the pattern frequency and the number of orientations. Grey = widefield passband, green = shifted copies (±m=1).</p>
   <div id="fourier" style="height:400px"></div>
   <div class="ctl" style="grid-template-columns:1fr 1fr;gap:18px;margin-top:8px"><div><label>Pattern frequency / cut-off <output id="fr_o"></output></label><input id="fr" type="range" min="0.3" max="1" step="0.01" value="0.85"></div>
    <div><label>Orientations <output id="no_o"></output></label><input id="no" type="range" min="1" max="3" step="1" value="3"></div></div>
   <div class="kpis" style="margin-top:14px;grid-template-columns:repeat(3,1fr)"><div class="kpi"><b id="m_area">—</b><span>passband area vs. widefield</span></div><div class="kpi"><b id="m_best">—</b><span>best-direction resolution gain</span></div><div class="kpi"><b id="m_worst">—</b><span>worst-direction resolution gain</span></div></div></div>
  <div class="card rv"><h3>Widefield vs. SIM</h3><p class="cap">Drag the slider · ideal calibration, 200 photons per frame (simulated from the ground truth)</p>
   <div class="seg" id="cmpSeg"></div>
   <div class="cmp" id="cmp"><img id="cmpB" alt="SIM"><img id="cmpA" alt="widefield"><div class="hd" id="cmpH"></div><span class="tag" style="left:12px;top:12px">Widefield</span><span class="tag" style="right:12px;bottom:12px">SIM</span></div>
   <input id="cmpR" type="range" min="0" max="100" value="50" style="margin-top:12px"><p class="small" id="cmpM" style="text-align:center;margin:0"></p></div>
 </div>
 <div class="card rv"><h3>The nine raw frames</h3><p class="cap">frames ordered by orientation (0°, 60°, 120°) then phase (0°, 120°, 240°) · 100 photons — the stripes are the information carrier</p><div class="raws9">@@RAW_HTML@@</div></div>
 <div class="card rv mt"><h3>Raw spectrum → reconstruction</h3><p class="cap">Yellow dots: the pattern frequencies ±p. Dashed grey: widefield cut-off. Dashed green: SIM support — the reconstruction fills the larger disc.</p>@@F_FOURIER@@</div>
</div></section>

<section class="sec" id="lab"><div class="wrap">
 <div class="sh rv"><div class="bd">3</div><div><h2>Interactive lab</h2><p>Every combination below was simulated and reconstructed in the notebook (@@N_LAB@@ conditions × 3 methods); the sliders look the results up. Try to break the classical reconstruction, then see what the local model and the learned module do about it.</p></div></div>
 <div class="card rv">
  <div class="ctl"><div><label>Photon budget <output id="o_n"></output></label><input id="s_n" type="range" min="0" max="@@NMAX@@" step="1" value="1"></div>
   <div><label>Pattern-phase error <output id="o_p"></output></label><input id="s_p" type="range" min="0" max="@@PMAX@@" step="1" value="0"></div>
   <div><label>Pattern-frequency error <output id="o_f"></output></label><input id="s_f" type="range" min="0" max="@@FMAX@@" step="1" value="0"></div></div>
  <div class="pre"><span class="small" style="align-self:center">Presets:</span><button onclick="preset(1,0,0)">Ideal</button><button onclick="preset(1,2,2)">Typical</button><button onclick="preset(2,3,3)">Hard</button><button onclick="preset(3,4,4)">Brutal</button></div>
  <div class="labgrid">
   <div class="pn"><h4><i style="background:#bdbebf"></i>Widefield</h4><img id="i_wf" alt="widefield"><div class="m"><b id="p_wf"></b> dB<br><span id="s_wf"></span></div></div>
   <div class="pn"><h4><i style="background:#494b52"></i>Classical SIM</h4><img id="i_cl" alt="classical"><div class="m"><b id="p_cl"></b> dB<br><span id="s_cl"></span></div></div>
   <div class="pn"><h4><i style="background:#268c32"></i>Local-pattern model</h4><img id="i_lo" alt="local"><div class="m"><b id="p_lo"></b> dB <span class="dl" id="d_lo"></span><br><span id="s_lo"></span></div></div>
   <div class="pn"><h4><i style="background:#e0d01f"></i>Classical + learned</h4><img id="i_nn" alt="learned"><div class="m"><b id="p_nn"></b> dB <span class="dl" id="d_nn"></span><br><span id="s_nn"></span></div></div>
   <div class="pn"><h4><i style="background:#1dad22"></i>Ground truth</h4><img id="i_gt" alt="ground truth"><div class="m"><b>∞</b> dB<br><span>reference (128² crop)</span></div></div>
  </div>
  <p class="small" style="margin:14px 0 0">Scenario: membranes image (cells3d), inhomogeneity η = 0.4, 10 % out-of-focus haze. Metrics are measured on the full 256² field; panels show a 128² region. Δ is relative to classical SIM.</p>
 </div>
 <div class="card mt rv"><h3>All @@N_LAB@@ conditions at once</h3><p class="cap">Parallel coordinates — drag along any axis to select a range. Line colour = gain of the learned module over classical SIM (green = better, grey = worse).</p>@@F_PARCOORDS@@</div>
</div></section>

<section class="sec alt" id="q1"><div class="wrap">
 <div class="sh rv"><div class="bd">4</div><div><span class="rq">RESEARCH QUESTION 1</span><h2>How sensitive is SIM to calibration errors — and can the data calibrate itself?</h2><p>Reconstruction assumes the pattern’s frequency and phase. I misstate them by controlled amounts (100 photons, three held-out images × @@SEEDS@@ error patterns) and compare with a self-calibration step that estimates both from the raw frames.</p></div></div>
 <div class="g2"><div class="card rv"><h3>PSNR landscape</h3><p class="cap">▾ marks cells below the widefield + Wiener level — there, SIM is worse than not using SIM</p>@@F_Q1HEAT@@</div>
  <div class="card rv"><h3>Same landscape in 3D</h3><p class="cap">Grey sheet = widefield + Wiener level · drag to rotate</p>@@F_Q1SURF@@</div></div>
 <div class="find rv">@@T_Q1A@@ @@T_Q1B@@</div>
 <div class="card rv"><h3>Self-calibration vs. photon budget</h3><p class="cap">True errors: 10° phase, 0.8 cycles/FOV frequency · markers = median, bars = inter-quartile range</p>@@F_Q1EST@@</div>
 <div class="find rv">@@T_Q1C@@</div>
 <div class="card mt rv"><h3>Where does the error live in Fourier space?</h3><p class="cap">Power spectrum of the reconstruction error. Mis-set pattern frequencies leave energy rings around the true pattern positions (yellow circles) — the “grid” artifacts.</p>@@F_Q1RES@@</div>
</div></section>

<section class="sec" id="q2"><div class="wrap">
 <div class="sh rv"><div class="bd">5</div><div><span class="rq">RESEARCH QUESTION 2</span><h2>Do smooth illumination distortions create artifacts — and does estimating them locally help?</h2><p>Real patterns are not perfect sinusoids: phase and contrast drift across the field. I warp the hidden pattern by a smooth random field of strength η (up to 1.2 rad and −50 % contrast, different for each orientation) and compare the global-sinusoid model with a local one that estimates the pattern’s complex gain from the data. Calibration is known here, to isolate the effect.</p></div></div>
 <div class="g2"><div class="card rv"><h3>Quality vs. distortion strength</h3><p class="cap">100 photons · mean ± s.d. over 3 images × @@SEEDS3@@ pattern realisations</p>@@F_Q2LINES@@</div>
  <div class="card rv"><h3>Why: the estimator sees the distortion</h3><p class="cap">Strongest distortion (η = 1), membranes, orientation 0°. Right: where the reconstruction error sits</p>@@F_Q2MAPS@@
   <div class="find" style="margin-bottom:0">@@T_Q2B@@</div></div></div>
 <div class="find rv">@@T_Q2A@@ This local model is my own simplified, illustrative take on flexible illumination modelling — not the FlexSIM algorithm.</div>
</div></section>

<section class="sec alt" id="q3"><div class="wrap">
 <div class="sh rv"><div class="bd">6</div><div><span class="rq">RESEARCH QUESTION 3</span><h2>Can a small learned module clean up what the classical pipeline leaves — and where does it fail?</h2><p>A 3-level residual U-Net post-processes the SIM reconstruction. It is trained on simulated reconstructions of <i>kidney and lily</i> images under random noise, calibration errors, distortions and haze, then tested on <i>cells3d and HeLa</i> images it has never seen — a cross-dataset test, not a held-out crop of the same sample.</p></div></div>
 <div class="g2"><div class="card rv"><h3>Training</h3><p class="cap">Validation error on unseen images falls below the classical input</p>@@F_Q3TRAIN@@<div class="find" style="margin-bottom:0">@@T_Q3A@@</div></div>
  <div class="card rv"><h3>Example (membranes)</h3><p class="cap">Same data, five pipelines · intensity-matched to ground truth</p><div class="raws" style="grid-template-columns:repeat(4,1fr)">@@EX_HTML@@</div></div></div>
 <div class="card rv"><h3>Held-out performance across photon budgets</h3><p class="cap">Realistic calibration condition: 10° phase, 0.4 cyc/FOV frequency, η = 0.4, 10 % haze · bars = mean, whiskers = s.d. over 3 images × @@SEEDS3@@ seeds</p>@@F_Q3BARS@@</div>
 <div class="find rv">@@T_Q3B@@</div>
 <div class="g2 mt"><div class="card rv"><h3>Before / after, condition by condition</h3><p class="cap">Best classical pipeline → plus learned module (green link = gain, dark = loss)</p>@@F_Q3DUMB@@</div>
  <div class="card rv"><h3>Does it invent detail?</h3><p class="cap">Radial spectra: a faithful method follows the ground truth; a smoothing one drops early</p>@@F_Q3SPEC@@</div></div>
 <div class="card rv"><h3>Reconstruction vs. truth, pixel by pixel</h3><p class="cap">Density of (truth, reconstruction) intensity pairs · the dashed diagonal is a perfect reconstruction</p>@@F_Q3HIST@@</div>
 <div class="card mt rv"><h3>Stress tests — out of distribution</h3><p class="cap">Haze beyond the training range, and calibration errors beyond it</p>@@F_Q3STRESS@@</div>
 <div class="find rv">@@T_Q3C@@</div>
</div></section>

<section class="sec" id="limits"><div class="wrap">
 <div class="sh rv"><div class="bd">7</div><div><h2>Method, honest limits, next steps</h2><p>Everything on this page was produced by one notebook with fixed seeds, from public images, in a free Colab session.</p></div></div>
 <div class="g2"><div class="card rv"><h3>Setup</h3><table class="p">@@PARAMS@@</table></div>
  <div class="card rv"><h3>What this is — and is not</h3><ul class="lim">
   <li><b>Simulated acquisitions.</b> Raw SIM frames are generated from real fluorescence images; the images already contain their own blur and noise. The next step is real SIM raw data (for example the public BioSR dataset).</li>
   <li><b>2D, single wavelength,</b> idealised OTF, fixed pattern frequency geometry.</li>
   <li><b>Simplified local model.</b> It illustrates why modelling spatial variation matters; it is not FlexSIM.</li>
   <li><b>Small learned module,</b> trained on two datasets for a few minutes. Gains are measured on the simulator, not on microscopes.</li>
   <li><b>Next:</b> learned calibration/prior inside the reconstruction loop rather than only as post-processing; real data; out-of-focus background from z-stacks.</li></ul></div></div>
</div></section>

<footer><div class="wrap">SIM Lab · built by @@AUTHOR@@ · @@DATE@@ · Public data: scikit-image sample images (cells3d, kidney, lily, HeLa mitosis). Inspired by the M2 internship proposal on SIM image reconstruction at IRIT / CNRS; no affiliation implied.</div></footer>

<script>
const LAB=@@LAB_JSON@@, CMP=@@CMP_JSON@@;
const $=id=>document.getElementById(id);
/* ---------- interactive lab ---------- */
const NL=LAB.peaks.length,PL=LAB.ph.length,FL=LAB.fq.length;
function lab(){
 const n=+$("s_n").value,p=+$("s_p").value,f=+$("s_f").value,i=(n*PL+p)*FL+f,m=LAB.m[i],w=LAB.wf_m[n];
 $("o_n").textContent=LAB.peaks[n]+" photons";$("o_p").textContent=LAB.ph[p]+"°";$("o_f").textContent=LAB.fq[f]+" cyc/FOV";
 $("i_wf").src=LAB.wf[n];$("i_cl").src=LAB.cl[i];$("i_lo").src=LAB.loc[i];$("i_nn").src=LAB.nn[i];$("i_gt").src=LAB.gt;
 $("p_wf").textContent=w[0].toFixed(1);$("s_wf").textContent="SSIM "+w[1].toFixed(2);
 $("p_cl").textContent=m[0].toFixed(1);$("s_cl").textContent="SSIM "+m[1].toFixed(2);
 $("p_lo").textContent=m[2].toFixed(1);$("s_lo").textContent="SSIM "+m[3].toFixed(2);
 $("p_nn").textContent=m[4].toFixed(1);$("s_nn").textContent="SSIM "+m[5].toFixed(2);
 [["d_lo",m[2]-m[0]],["d_nn",m[4]-m[0]]].forEach(([id,d])=>{const e=$(id);e.textContent="("+(d>=0?"+":"−")+Math.abs(d).toFixed(1)+")";e.style.color=d>=0?"#1dad22":"#494b52"});
}
function preset(n,p,f){$("s_n").value=Math.min(n,NL-1);$("s_p").value=Math.min(p,PL-1);$("s_f").value=Math.min(f,FL-1);lab()}
["s_n","s_p","s_f"].forEach(id=>$(id).addEventListener("input",lab));preset(1,0,0);
/* ---------- widefield vs SIM slider ---------- */
const names=Object.keys(CMP);let cur=names[0];
names.forEach(nm=>{const b=document.createElement("button");b.textContent=nm;b.onclick=()=>{cur=nm;show()};b.dataset.n=nm;$("cmpSeg").appendChild(b)});
function show(){const d=CMP[cur];$("cmpA").src=d.wf;$("cmpB").src=d.sim;$("cmpM").textContent=d.src+" · widefield "+d.psnr[0].toFixed(1)+" dB → SIM "+d.psnr[1].toFixed(1)+" dB";
 [...$("cmpSeg").children].forEach(b=>b.classList.toggle("on",b.dataset.n===cur));cut()}
function cut(){const v=+$("cmpR").value;$("cmpA").style.clipPath="inset(0 "+(100-v)+"% 0 0)";$("cmpH").style.left=v+"%"}
$("cmpR").addEventListener("input",cut);show();
/* ---------- Fourier explorer ---------- */
function fourier(){
 const r=+$("fr").value,no=+$("no").value;$("fr_o").textContent=r.toFixed(2);$("no_o").textContent=no;
 const th=[0,60,120].slice(0,no).map(d=>d*Math.PI/180),c=[[0,0]];
 th.forEach(t=>{c.push([r*Math.cos(t),r*Math.sin(t)]);c.push([-r*Math.cos(t),-r*Math.sin(t)])});
 const inU=(x,y)=>c.some(q=>(x-q[0])**2+(y-q[1])**2<=1);
 let cnt=0;for(let x=-2.1;x<=2.1;x+=0.03)for(let y=-2.1;y<=2.1;y+=0.03)if(inU(x,y))cnt++;
 let c0=0;for(let x=-1;x<=1;x+=0.03)for(let y=-1;y<=1;y+=0.03)if(x*x+y*y<=1)c0++;
 let mn=9,mx=0;for(let a=0;a<360;a+=3){const t=a*Math.PI/180;let q=0;while(q<2.2&&inU(q*Math.cos(t),q*Math.sin(t)))q+=0.01;mn=Math.min(mn,q);mx=Math.max(mx,q)}
 $("m_area").textContent="×"+(cnt/c0).toFixed(2);$("m_best").textContent="×"+mx.toFixed(2);$("m_worst").textContent="×"+mn.toFixed(2);
 const shapes=c.map((q,i)=>({type:"circle",xref:"x",yref:"y",x0:q[0]-1,x1:q[0]+1,y0:q[1]-1,y1:q[1]+1,line:{color:i?"#1dad22":"#494b52",width:i?1.6:2.2},fillcolor:i?"rgba(29,173,34,0.10)":"rgba(73,75,82,0.14)"}));
 Plotly.react("fourier",[{x:c.map(q=>q[0]),y:c.map(q=>q[1]),mode:"markers",marker:{color:c.map((q,i)=>i?"#e0d01f":"#494b52"),size:9,line:{color:"#494b52",width:1}},hoverinfo:"skip"}],
  {shapes:shapes,margin:{l:40,r:10,t:10,b:40},paper_bgcolor:"rgba(0,0,0,0)",plot_bgcolor:"rgba(0,0,0,0)",showlegend:false,font:{family:"Inter,system-ui,sans-serif",color:"#494b52"},
   xaxis:{range:[-2.3,2.3],title:"kx / cut-off",gridcolor:"#eceded",zeroline:false,scaleanchor:"y"},yaxis:{range:[-2.3,2.3],title:"ky / cut-off",gridcolor:"#eceded",zeroline:false}},{displayModeBar:false,responsive:true});
}
["fr","no"].forEach(id=>$(id).addEventListener("input",fourier));fourier();
/* ---------- nav + reveal ---------- */
const secs=[...document.querySelectorAll("section")],links=[...document.querySelectorAll("nav a")];
if("IntersectionObserver" in window){
 const io=new IntersectionObserver(es=>es.forEach(e=>{if(e.isIntersecting){e.target.classList.add("in");io.unobserve(e.target)}}),{threshold:0.05});
 document.querySelectorAll(".rv").forEach(el=>io.observe(el));
 const so=new IntersectionObserver(es=>es.forEach(e=>{if(e.isIntersecting)links.forEach(a=>a.classList.toggle("on",a.getAttribute("href")==="#"+e.target.id))}),{rootMargin:"-40% 0px -55% 0px"});
 secs.forEach(s=>so.observe(s));
}else document.querySelectorAll(".rv").forEach(el=>el.classList.add("in"));
if(/[?&]all/.test(location.search)||(window.matchMedia&&matchMedia("print").matches)){document.documentElement.classList.add("noanim");document.querySelectorAll(".rv").forEach(el=>el.classList.add("in"))}
setTimeout(()=>document.querySelectorAll(".rv").forEach(el=>{if(el.getBoundingClientRect().top<innerHeight*1.1)el.classList.add("in")}),900);
</script></body></html>'''

ex_html = "".join(f'<figure><img src="{R["q3_examples"][k]}" alt="{k}"><figcaption>{k}</figcaption></figure>' for k in ["Ground truth"] + METHODS)
TOK = dict(PLOTLY=plotly.offline.get_plotlyjs(), AUTHOR=AUTHOR, DATE=time.strftime("%d %B %Y"), DEVICE=("T4 GPU" if DEV == "cuda" else "CPU") if DEV != "cuda" else torch.cuda.get_device_name(0),
           RUNTIME=f"{(time.time()-T_START)/60:.0f}", HERO_SIM=CMP["Membranes"]["sim"], HERO_WF=CMP["Membranes"]["wf"],
           K1=f"{K['support']:.2f}", K2=sgn(K["sim_gain"]), K3=f"{K['auto_err']:.2f}", K3P=str(int(pk_ref)), K4=sgn(K["learned"]),
           N_ALL=str(len(INV)), N_KEPT=str(n_kept), N_EXC=str(n_exc), N_LAB=str(len(LAB["cl"])), NMAX=str(len(LAB["peaks"]) - 1), PMAX=str(len(LAB["ph"]) - 1), FMAX=str(len(LAB["fq"]) - 1),
           SEEDS=str(SEEDS), SEEDS3=str(SEEDS + 1), TEST_HTML=test_html, RAW_HTML=raw_html, EX_HTML=ex_html, PARAMS=params_html,
           LAB_JSON=json.dumps(LAB), CMP_JSON=json.dumps(CMP),
           T_Q1A=T["q1a"], T_Q1B=T["q1b"], T_Q1C=T["q1c"], T_Q2A=T["q2a"], T_Q2B=T["q2b"], T_Q3A=T["q3a"], T_Q3B=T["q3b"], T_Q3C=T["q3c"])
TOK.update({"F_" + k.upper().replace("_", ""): v for k, v in F_.items()})
html = TEMPLATE
for k, v in TOK.items(): html = html.replace(f"@@{k}@@", v)
left = sorted(set(__import__("re").findall(r"@@([A-Z0-9_]+)@@", html)))
assert not left, f"unfilled placeholders: {left}"
out_path = os.path.join(OUT_DIR, OUT_NAME)
with open(out_path, "w", encoding="utf-8") as fh: fh.write(html)
print(f"\nDashboard saved: {out_path}  ({os.path.getsize(out_path)/1e6:.1f} MB) | total runtime {(time.time()-T_START)/60:.1f} min")
if IN_COLAB:
    try:
        from google.colab import files; files.download(out_path)
    except Exception as e: print("download skipped:", e)



Dashboard saved: /content/drive/MyDrive/SIM_Lab/sim_lab_dashboard.html  (8.0 MB) | total runtime 10.8 min


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>